# FIT5196 Assessment 1 - Solution Notebook

**Group:** Group005  
**Members:** Replace with names and student IDs

## 0. Configuration and reproducibility

### Scope — Tasks 1, 2, 3 and 4

This notebook covers **specification Tasks 1, 2, 3 and 4**:

| Rubric criterion | Marks | Covered here |
|---|---|---|
| A1. Structured parsing and source profiling | 0.9 | sections `T1-P1` … `T1-P9` |
| A2. Source-to-target mapping | 0.6 | section `T1-M1` |
| B1–B3. Core relational transformation | 3.0 | sections `T2-R1` … `T2-O1` |
| C1–C2. Reconciliation/integrity evidence used here | 1.5 | `T2-R1` and validation register |
| E1–E2. Validation/reproducibility evidence | 2.0 | validation register (`VAL-*`) |

Tasks 5 and 6 are **not** implemented here. Section headings carry stable evidence IDs
(`T1-P1` … `T1-M1`) so the report and the source-to-target mapping cite them without copying
the evidence.

**Method.** Following the unit's applied sessions, the JSON export is read with `json` and
flattened with `pandas.json_normalize` (Week 3), the XML export is read with
`BeautifulSoup(..., "lxml-xml")` (Week 2), and every profiling question is then answered with
**pandas** on those DataFrames.

**Where regular expressions are and are not used.** The specification draws a hard line:
*"Do not parse either structured document as plain text with regular expressions"* (Task 1),
while regex is the assessed tool for *"bounded patterns such as tags, markers, URLs,
whitespace and business-reference extraction"* applied **after** a structured parser has
returned the value (Task 3).

This notebook keeps that line visible:

| Stage | Tool | Regex? |
|---|---|---|
| Reading the JSON document | `json.load` | no |
| Reading the XML document | `pandas.read_xml` (lxml), `BeautifulSoup("lxml-xml")` for the tree walk | no |
| Flattening to DataFrames | `pandas.json_normalize`, `pandas.read_xml` | no |
| Grain, keys, duplicates, overlap, foreign keys | pandas | no |
| Currency / whitespace / digit-shape handling | `str.split`, `str.replace(regex=False)`, `str.translate` | **no** |
| `T1-P8` narrative markers, tags and business references | `re` / `pandas.str`, verbose `(?x)` | yes — as Task 3 requires |

`T1-P8` is the **only** section that uses a regular expression, and it operates on values the
structured parsers already returned. A check at the end of the notebook verifies this.

No canonical row count, expected value or duplicate-id list is hard-coded anywhere.

### 0.1 Environment and dependencies

---
#### Imports

In [1]:
import glob
import hashlib
import html
import json
import os
import re
import sys
import unicodedata
from collections import Counter

import pandas as pd
from bs4 import BeautifulSoup as bsoup

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)

print("python :", sys.version.split()[0])
print("pandas :", pd.__version__)

python : 3.11.5
pandas : 2.0.3


---
#### Configuration cell

All paths are relative to the notebook directory and can be overridden with environment
variables, so the workflow runs from a fresh kernel with **Restart and Run All** without
editing code.

In [2]:
GROUP_ID = "Group005"
INPUT_DIR = os.environ.get("FIT5196_INPUT_DIR", "raw_input")
OUTPUT_DIR = os.environ.get("FIT5196_OUTPUT_DIR", "outputs")
REFERENCE_DIR = os.environ.get("FIT5196_REFERENCE_DIR", "reference")
TEMPLATE_DIR = os.environ.get("FIT5196_TEMPLATE_DIR", "templates")
PROFILE_DIR = os.environ.get("FIT5196_PROFILE_DIR", "profiling")

JSON_PATH = os.path.join(INPUT_DIR, f"{GROUP_ID}_commerce.json")
XML_PATH = os.path.join(INPUT_DIR, f"{GROUP_ID}_operations.xml")
DICTIONARY_PATH = os.path.join(REFERENCE_DIR, "public_data_dictionary.csv")
MANIFEST_PATH = os.path.join(REFERENCE_DIR, "A1_manifest.json")
MAPPING_PATH = f"{GROUP_ID}_source_to_target_mapping.csv"

for directory in (OUTPUT_DIR, PROFILE_DIR):
    os.makedirs(directory, exist_ok=True)

MISSING = "NaN"          # published literal missing-string sentinel
MONEY_TOLERANCE = 0.01   # published comparison tolerance for monetary fields

def save_profile(frame, name):
    '''Write a Task 1 profiling artefact and echo where it landed.'''
    path = os.path.join(PROFILE_DIR, f"{GROUP_ID}_T1_{name}.csv")
    frame.to_csv(path, index=False)
    print(f"[saved] {path}  ({len(frame)} rows)")
    return path

for label, path in [("json", JSON_PATH), ("xml", XML_PATH), ("dictionary", DICTIONARY_PATH)]:
    print(f"{label:11s} {path}  exists={os.path.exists(path)}")

json        raw_input/Group005_commerce.json  exists=True
xml         raw_input/Group005_operations.xml  exists=True
dictionary  reference/public_data_dictionary.csv  exists=True


## 1. Parse and profile the two sources

### 1.1 JSON structure and profile

---
#### `T1-P1` — Structured parsing and file integrity

The JSON export is read with the standard-library `json` parser and the XML export with
`BeautifulSoup` using the `lxml-xml` parser, exactly as in the Week 2 and Week 3 applied
sessions. No regular expression touches document structure.

File integrity is checked against the published `A1_manifest.json` SHA-256 digests, so a
truncated or substituted input is detected before any conclusion is drawn.

In [3]:
def sha256_of(path, chunk=1 << 20):
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(chunk), b""):
            digest.update(block)
    return digest.hexdigest()

with open(MANIFEST_PATH) as fh:
    manifest = json.load(fh)
expected = {os.path.basename(entry["path"]): entry for entry in manifest["files"]}

integrity = pd.DataFrame([
    {
        "file": os.path.basename(path),
        "bytes_observed": os.path.getsize(path),
        "bytes_manifest": expected.get(os.path.basename(path), {}).get("bytes"),
        "sha256_matches_manifest": sha256_of(path) == expected.get(os.path.basename(path), {}).get("sha256"),
    }
    for path in (JSON_PATH, XML_PATH)
])
print("manifest group alias:", manifest.get("group_alias"), "| notebook GROUP_ID:", GROUP_ID)
integrity

manifest group alias: Group005 | notebook GROUP_ID: Group005


,file,bytes_observed,bytes_manifest,sha256_matches_manifest
0,Group005_commerce.json,14139534,14139534,True
1,Group005_operations.xml,17811170,17811170,True


In [4]:
# --- JSON: json.load, then pandas (Week 3 applied session) ------------------
with open(JSON_PATH, encoding="utf-8") as fh:
    commerce = json.load(fh)

print("JSON top level  :", type(commerce).__name__)
for key, value in commerce.items():
    print(f"   {key:20s} {type(value).__name__:6s} {len(value) if hasattr(value, '__len__') else ''}")
print("JSON exportMetadata :", commerce["exportMetadata"])

JSON top level  : dict
   customerProfiles     list   500
   exportMetadata       dict   3
   orders               list   2818
   productReviews       list   3946
JSON exportMetadata : {'groupAlias': 'Group005', 'period': 2018, 'sourceSystem': 'CommercePlatform'}


### 1.2 XML structure and profile

In [5]:
# --- XML: BeautifulSoup with the lxml-xml parser (Week 2 applied session) ---
with open(XML_PATH, encoding="utf-8") as fh:
    operations = bsoup(fh, "lxml-xml")

root = operations.find("OperationsExport")
print("XML root        :", root.name, dict(root.attrs))
for child in root.find_all(recursive=False):
    print(f"   {child.name:22s} {len(child.find_all(recursive=False)):5d} direct children")
print("XML Export_Metadata :",
      {c.name: c.text for c in root.find("Export_Metadata").find_all(recursive=False)})

XML root        : OperationsExport {'groupAlias': 'Group005', 'sourceSystem': 'OperationsERP', 'period': '2018'}
   Export_Metadata            2 direct children
   Orders                  2818 direct children
   ProductCatalogue        1000 direct children
   ProductReviews          3946 direct children
   WarehouseDirectory         3 direct children
XML Export_Metadata : {'System': 'OperationsERP', 'Contract_Version': 'A1-DATA-2.0'}


### 1.3 Source comparison and assumptions

**Observation.** Both exports declare the same group alias (`Group005`) and the same reporting
period (2018) but a different source system (`CommercePlatform` vs `OperationsERP`). The two
systems describe overlapping business events with different nesting, element names and value
conventions — which is what the rest of Task 1 quantifies.

---
#### `T1-P2` — Flattening to DataFrames, and the structural profile

Every source collection is flattened into a **pandas DataFrame**: JSON with
`pandas.json_normalize` (including `record_path` for the nested cart array), XML with
`pandas.read_xml` and an XPath per collection. Both are real structured parsers — `read_xml`
runs `lxml` underneath — so no document is scraped as text.

The eleven frames below are the single tabular representation used from `T1-P3` onwards, and
the hand-off point into Task 2. Each one is displayed so the actual parsed values are visible,
not just their shape.

Source typing is deliberately **not** coerced here: JSON frames keep native JSON types
(`bool`/`int`/`float`/`str`), XML frames are entirely `object`/`str` because XML has no value
types. That difference is itself the finding profiled in `T1-P4`.

In [6]:
# --- JSON -> DataFrame with pandas.json_normalize --------------------------
orders_flat = pd.json_normalize(commerce["orders"])          # header.* and delivery.* columns

json_headers = (orders_flat.filter(like="header.")
                .rename(columns=lambda c: c.split(".", 1)[1]))
json_deliveries = (orders_flat.filter(like="delivery.")
                   .rename(columns=lambda c: c.split(".", 1)[1]))
json_items = pd.json_normalize(commerce["orders"], record_path="shoppingCart")
json_customers = pd.json_normalize(commerce["customerProfiles"])
json_reviews = pd.json_normalize(commerce["productReviews"])

for name, frame in [("json_customers", json_customers), ("json_headers", json_headers),
                    ("json_items", json_items), ("json_deliveries", json_deliveries),
                    ("json_reviews", json_reviews)]:
    print(f"  {name:16s} {frame.shape}")

  json_customers   (500, 20)
  json_headers     (2818, 22)
  json_items       (8947, 6)
  json_deliveries  (2818, 20)
  json_reviews     (3946, 15)


In [7]:
# --- XML -> DataFrame with pandas.read_xml -----------------------------------
# read_xml is pandas' own XML reader (lxml underneath), so the document is still
# handled by a real XML parser - no text scraping. dtype=str is essential: without
# it read_xml infers numeric dtypes and silently rewrites 13 columns, which would
# both destroy the "XML publishes everything as text" finding profiled in T1-P4 and
# risk stripping leading zeros from any all-digit identifier.
XML_COLLECTIONS = {
    "xml_headers":    ".//Order/Header",
    "xml_items":      ".//Order/Shopping_Cart/Item",
    "xml_deliveries": ".//Order/Delivery",
    "xml_products":   ".//ProductCatalogue/Product",
    "xml_reviews":    ".//ProductReviews/Review",
    "xml_warehouses": ".//WarehouseDirectory/Warehouse",
}

xml_frames = {name: pd.read_xml(XML_PATH, xpath=xpath, dtype=str)
              for name, xpath in XML_COLLECTIONS.items()}

xml_headers = xml_frames["xml_headers"]
xml_items = xml_frames["xml_items"]
xml_deliveries = xml_frames["xml_deliveries"]
xml_products = xml_frames["xml_products"]
xml_reviews = xml_frames["xml_reviews"]
xml_warehouses = xml_frames["xml_warehouses"]

for name, frame in xml_frames.items():
    print(f"  {name:16s} {frame.shape}")

  xml_headers      (2818, 22)
  xml_items        (8803, 6)
  xml_deliveries   (2818, 20)
  xml_products     (1000, 21)
  xml_reviews      (3946, 15)
  xml_warehouses   (3, 3)


In [8]:
# Show the actual parsed data, not just its shape. Every downstream conclusion in
# this notebook is drawn from these eleven frames, so they are displayed here.
RAW_FRAMES = {
    ("JSON", "$.customerProfiles[*]"):                              json_customers,
    ("JSON", "$.orders[*].header"):                                 json_headers,
    ("JSON", "$.orders[*].shoppingCart[*]"):                        json_items,
    ("JSON", "$.orders[*].delivery"):                               json_deliveries,
    ("JSON", "$.productReviews[*]"):                                json_reviews,
    ("XML",  "/OperationsExport/Orders/Order/Header"):              xml_headers,
    ("XML",  "/OperationsExport/Orders/Order/Shopping_Cart/Item"):  xml_items,
    ("XML",  "/OperationsExport/Orders/Order/Delivery"):            xml_deliveries,
    ("XML",  "/OperationsExport/ProductCatalogue/Product"):         xml_products,
    ("XML",  "/OperationsExport/ProductReviews/Review"):            xml_reviews,
    ("XML",  "/OperationsExport/WarehouseDirectory/Warehouse"):     xml_warehouses,
}

for (source, path), frame in RAW_FRAMES.items():
    print(f"\n{'=' * 100}\n{source}  {path}      {frame.shape[0]:,} rows x {frame.shape[1]} columns\n{'=' * 100}")
    print(frame.head())


JSON  $.customerProfiles[*]      500 rows x 20 columns
  accountStatus acquisitionSource ageBand contactFrequencyPreference customerID customerSegment  emailDomain homeCountry homePostcode homeState homeSuburb householdSizeBand  \
0        Active             Store   25-34                  Quarterly   CUS00001           Value  example.net   Australia         3182       VIC   St Kilda                5+   
1        Active       Paid Search     55+             Essential only   CUS00002  Small Business    mail.test   Australia         3070       VIC  Northcote                5+   
2        Active            Social   35-44                    Monthly   CUS00003           Value  example.net   Australia         3070       VIC  Northcote                5+   
3        Review          Referral     55+                  Quarterly   CUS00004  Small Business  example.net   Australia         3121       VIC   Richmond               3-4   
4        Active          Referral   25-34                     We

In [9]:
raw_frame_profile = pd.DataFrame([
    {
        "source": source,
        "structural_path": path,
        "rows": len(frame),
        "columns": frame.shape[1],
        "dtypes": ", ".join(f"{kind}:{count}" for kind, count
                            in frame.dtypes.astype(str).value_counts().items()),
        "cells_missing": int(frame.isna().sum().sum()),
    }
    for (source, path), frame in RAW_FRAMES.items()
])
save_profile(raw_frame_profile, "raw_frame_profile")
raw_frame_profile

[saved] profiling/Group005_T1_raw_frame_profile.csv  (11 rows)


,source,structural_path,rows,columns,dtypes,cells_missing
0,JSON,$.customerProfiles[*],500,20,"object:17, float64:1, bool:1, int64:1",0
1,JSON,$.orders[*].header,2818,22,"object:14, float64:6, int64:1, bool:1",0
2,JSON,$.orders[*].shoppingCart[*],8947,6,"object:3, float64:2, int64:1",0
3,JSON,$.orders[*].delivery,2818,20,"object:11, int64:4, float64:3, bool:2",0
4,JSON,$.productReviews[*],3946,15,"object:12, int64:2, bool:1",0
5,XML,/OperationsExport/Orders/Order/Header,2818,22,object:22,1792
6,XML,/OperationsExport/Orders/Order/Shopping_Cart/Item,8803,6,object:6,0
7,XML,/OperationsExport/Orders/Order/Delivery,2818,20,object:20,0
8,XML,/OperationsExport/ProductCatalogue/Product,1000,21,object:21,0
9,XML,/OperationsExport/ProductReviews/Review,3946,15,object:15,0


In [10]:
# --- structural path profile (on the parse tree, where nesting still exists) ---
def profile_json_paths(node, path="$", acc=None):
    '''Recursively enumerate JSON structural paths with counts and node kind.'''
    acc = {"count": Counter(), "kind": {}, "types": {}} if acc is None else acc
    acc["count"][path] += 1
    acc["types"].setdefault(path, Counter())
    if isinstance(node, dict):
        acc["kind"][path] = "container"
        acc["types"][path]["object"] += 1
        for key, value in node.items():
            profile_json_paths(value, f"{path}.{key}", acc)
    elif isinstance(node, list):
        acc["kind"][path] = "container"
        acc["types"][path]["array"] += 1
        for element in node:
            profile_json_paths(element, f"{path}[*]", acc)
    else:
        acc["kind"].setdefault(path, "leaf")
        acc["types"][path][type(node).__name__] += 1
    return acc

json_acc = profile_json_paths(commerce)
json_paths = pd.DataFrame([
    {"source": "JSON", "structural_path": path, "occurrences": count,
     "node_kind": json_acc["kind"][path],
     "node_types": ", ".join(f"{k}:{v}" for k, v in json_acc["types"][path].most_common())}
    for path, count in json_acc["count"].items()
])

def profile_xml_paths(element, prefix="", acc=None):
    acc = {"count": Counter(), "kind": {}} if acc is None else acc
    path = f"{prefix}/{element.name}"
    acc["count"][path] += 1
    children = element.find_all(recursive=False)
    acc["kind"][path] = "container" if children else "leaf"
    for child in children:
        profile_xml_paths(child, path, acc)
    return acc

xml_acc = profile_xml_paths(root)
xml_paths = pd.DataFrame([
    {"source": "XML", "structural_path": path, "occurrences": count,
     "node_kind": xml_acc["kind"][path],
     "node_types": "element container" if xml_acc["kind"][path] == "container" else "text leaf"}
    for path, count in xml_acc["count"].items()
])

structure_profile = (pd.concat([json_paths, xml_paths], ignore_index=True)
                     .sort_values(["source", "structural_path"], ignore_index=True))
save_profile(structure_profile, "source_structure_profile")
print(f"JSON distinct paths: {len(json_paths)} | XML distinct paths: {len(xml_paths)}")

containers = structure_profile.query("node_kind == 'container'").reset_index(drop=True)
print(f"containers: JSON {(containers['source'] == 'JSON').sum()}, "
      f"XML {(containers['source'] == 'XML').sum()}")
containers

[saved] profiling/Group005_T1_source_structure_profile.csv  (201 rows)
JSON distinct paths: 98 | XML distinct paths: 103
containers: JSON 12, XML 14


,source,structural_path,occurrences,node_kind,node_types
0,JSON,$,1,container,object:1
1,JSON,$.customerProfiles,1,container,array:1
2,JSON,$.customerProfiles[*],500,container,object:500
3,JSON,$.exportMetadata,1,container,object:1
4,JSON,$.orders,1,container,array:1
5,JSON,$.orders[*],2818,container,object:2818
6,JSON,$.orders[*].delivery,2818,container,object:2818
7,JSON,$.orders[*].header,2818,container,object:2818
8,JSON,$.orders[*].shoppingCart,2818,container,array:2818
9,JSON,$.orders[*].shoppingCart[*],8947,container,object:8947


**Material structural findings**

* **JSON** nests three collections under one object: `customerProfiles[*]`, `orders[*]` (each
  with a `header` object, a `delivery` object and a **repeated `shoppingCart[*]` array**) and
  `productReviews[*]`. The order grain is therefore *not* the record grain for cart lines —
  the array is flattened with `json_normalize(..., record_path="shoppingCart")` to reach the
  `order_items` grain.
* **XML** nests four collections under `OperationsExport`: `Orders/Order` (with `Header`, a
  **repeated `Shopping_Cart/Item`** element and `Delivery`), `ProductCatalogue/Product`,
  `ProductReviews/Review` and `WarehouseDirectory/Warehouse`.
* **Entity coverage differs by source.** Customers exist only in JSON; the product catalogue
  exists only in XML. Orders, order items, deliveries and reviews exist in both. This drives
  every `source_format` value in the mapping.
* **`WarehouseDirectory` and the two metadata blocks map to no target table.** They are read
  and used as cross-checks but carry no mapping row.

---
#### `T1-P3` — Grain, candidate primary keys and candidate foreign keys

Each collection is tested against its candidate business key with pandas: row count, distinct
key count, and therefore whether the key is unique *within that source*.

In [11]:
COLLECTIONS = [
    # (source, structural path, frame, candidate key, target table, stated grain)
    ("JSON", "$.orders[*].header",                                json_headers,    "orderID",      "orders",          "one row per canonical order"),
    ("XML",  "/OperationsExport/Orders/Order/Header",             xml_headers,     "Order_ID",     "orders",          "one row per canonical order"),
    ("JSON", "$.orders[*].shoppingCart[*]",                       json_items,      "orderItemID",  "order_items",     "one row per order item"),
    ("XML",  "/OperationsExport/Orders/Order/Shopping_Cart/Item", xml_items,       "Order_Item_ID","order_items",     "one row per order item"),
    ("JSON", "$.customerProfiles[*]",                             json_customers,  "customerID",   "customers",       "one row per customer"),
    ("JSON", "$.orders[*].delivery",                              json_deliveries, "deliveryID",   "deliveries",      "one row per completed order delivery"),
    ("XML",  "/OperationsExport/Orders/Order/Delivery",           xml_deliveries,  "Delivery_ID",  "deliveries",      "one row per completed order delivery"),
    ("XML",  "/OperationsExport/ProductCatalogue/Product",        xml_products,    "Product_ID",   "products",        "one row per product"),
    ("JSON", "$.productReviews[*]",                               json_reviews,    "reviewID",     "product_reviews", "one row per canonical product review"),
    ("XML",  "/OperationsExport/ProductReviews/Review",           xml_reviews,     "Review_ID",    "product_reviews", "one row per canonical product review"),
]

grain_profile = pd.DataFrame([
    {
        "source": source,
        "structural_path": path,
        "target_table": table,
        "target_grain": grain,
        "records": len(frame),
        "distinct_keys": frame[key].nunique(dropna=False),
        "repeated_key_rows": len(frame) - frame[key].nunique(dropna=False),
        "key_unique_within_source": bool(frame[key].is_unique),
        "key_null_or_blank": int(frame[key].isna().sum() + (frame[key] == "").sum()),
    }
    for source, path, frame, key, table, grain in COLLECTIONS
])
save_profile(grain_profile, "source_grain_and_keys")
grain_profile

[saved] profiling/Group005_T1_source_grain_and_keys.csv  (10 rows)


,source,structural_path,target_table,target_grain,records,distinct_keys,repeated_key_rows,key_unique_within_source,key_null_or_blank
0,JSON,$.orders[*].header,orders,one row per canonical order,2818,2750,68,False,0
1,XML,/OperationsExport/Orders/Order/Header,orders,one row per canonical order,2818,2750,68,False,0
2,JSON,$.orders[*].shoppingCart[*],order_items,one row per order item,8947,8723,224,False,0
3,XML,/OperationsExport/Orders/Order/Shopping_Cart/Item,order_items,one row per order item,8803,8618,185,False,0
4,JSON,$.customerProfiles[*],customers,one row per customer,500,500,0,True,0
5,JSON,$.orders[*].delivery,deliveries,one row per completed order delivery,2818,2750,68,False,0
6,XML,/OperationsExport/Orders/Order/Delivery,deliveries,one row per completed order delivery,2818,2750,68,False,0
7,XML,/OperationsExport/ProductCatalogue/Product,products,one row per product,1000,1000,0,True,0
8,JSON,$.productReviews[*],product_reviews,one row per canonical product review,3946,3850,96,False,0
9,XML,/OperationsExport/ProductReviews/Review,product_reviews,one row per canonical product review,3946,3850,96,False,0


In [12]:
# Candidate foreign keys observed in the raw sources (before any reconciliation).
fk_candidates = pd.DataFrame([
    ("orders.customer_id",            "customers.customer_id",     "JSON header.customerID | XML Header/Customer_ID"),
    ("order_items.order_id",          "orders.order_id",           "item element repeats the parent order id in both sources"),
    ("order_items.product_id",        "products.product_id",       "XML ProductCatalogue is the only product source"),
    ("deliveries.order_id",           "orders.order_id",           "delivery block repeats the parent order id in both sources"),
    ("product_reviews.order_id",      "orders.order_id",           "review carries the order id directly"),
    ("product_reviews.order_item_id", "order_items.order_item_id", "review is tied to a single purchased line"),
    ("product_reviews.product_id",    "products.product_id",       "review carries the product id directly"),
    ("product_reviews.customer_id",   "customers.customer_id",     "review carries the customer id directly"),
], columns=["child_field", "parent_field", "source_evidence"])
fk_candidates

,child_field,parent_field,source_evidence
0,orders.customer_id,customers.customer_id,JSON header.customerID | XML Header/Customer_ID
1,order_items.order_id,orders.order_id,item element repeats the parent order id in both sources
2,order_items.product_id,products.product_id,XML ProductCatalogue is the only product source
3,deliveries.order_id,orders.order_id,delivery block repeats the parent order id in both sources
4,product_reviews.order_id,orders.order_id,review carries the order id directly
5,product_reviews.order_item_id,order_items.order_item_id,review is tied to a single purchased line
6,product_reviews.product_id,products.product_id,review carries the product id directly
7,product_reviews.customer_id,customers.customer_id,review carries the customer id directly


**Material grain findings**

* Every candidate key is fully populated — no null or blank identifier anywhere.
* **None of the four shared transactional collections is unique on its own key.** Orders,
  order items, deliveries and reviews each repeat a subset of their own records verbatim in
  *both* exports, so within-source duplication must be resolved before the cross-source
  overlap is even considered (`T1-P6`). The two single-source collections are the exception:
  `customerProfiles` (500/500) and `ProductCatalogue` (1,000/1,000) are already unique.
* `shoppingCart` / `Shopping_Cart/Item` is the only one-to-many child of an order. Flattening
  it produces the `order_items` grain; the parent order grain is untouched, so order-level
  money is never multiplied by the number of lines.
* Every order in **both** exports has `Order_Status = Completed` and exactly one delivery
  block, so `deliveries` is 1:1 with `orders` in this package rather than a strict subset.
  The grain is satisfied without filtering here; an explicit completed-order filter is **not**
  written in this Task 1 notebook and must be added in Task 2.

---
#### `T1-P4` — Value-format conventions

The two systems publish the same business values in different notations. Each convention is
probed with pandas over the **flattened DataFrame columns**, so the label and the evidence
cannot drift apart: every field named in `target_fields_probed` is actually probed.

In [13]:
# Digit masking uses str.translate, not a regular expression: nothing in the parsing
# or normalisation path needs regex, and keeping it out makes the boundary unambiguous.
DIGIT_MASK = str.maketrans("0123456789", "9999999999")

def pattern_counts(values, top=4):
    '''Digit-masked shape summary of a Series, e.g. 'AUD 9,999.99'.'''
    shapes = Counter()
    for value in values:
        if value is None or value == "" or (isinstance(value, float) and pd.isna(value)):
            shapes["<missing>"] += 1   # one bucket for all three absence encodings
        else:
            shapes[str(value).translate(DIGIT_MASK)[:28]] += 1
    return " | ".join(f"{shape} ({count})" for shape, count in shapes.most_common(top))

def columns_of(*specs):
    '''Stack one or more DataFrame columns into a single Series for profiling.'''
    return pd.concat([frame[column].astype(object) for frame, column in specs],
                     ignore_index=True)

conventions = [
    ("date",
     "customers.signup_date | deliveries.dispatch_date | deliveries.promised_date | "
     "deliveries.delivered_date | products.launch_date",
     columns_of((json_customers, "signupDate"), (json_deliveries, "dispatchDate"),
                (json_deliveries, "promisedDate"), (json_deliveries, "deliveredDate")),
     columns_of((xml_deliveries, "Dispatch_Date"), (xml_deliveries, "Promised_Date"),
                (xml_deliveries, "Delivered_Date"), (xml_products, "Launch_Date")),
     "YYYY-MM-DD"),
    ("timestamp",
     "orders.order_timestamp | product_reviews.review_timestamp",
     columns_of((json_headers, "orderTimestamp"), (json_reviews, "reviewTimestamp")),
     columns_of((xml_headers, "Order_Timestamp"), (xml_reviews, "Review_Timestamp")),
     "YYYY-MM-DD HH:MM:SS"),
    ("boolean",
     "orders.expedited_delivery | deliveries.on_time_in_full | deliveries.signature_required | "
     "customers.marketing_consent (JSON only) | products.recyclable_packaging, products.active_flag "
     "(XML only) | product_reviews.verified_purchase",
     columns_of((json_headers, "expeditedDelivery"), (json_deliveries, "onTimeInFull"),
                (json_deliveries, "signatureRequired"), (json_customers, "marketingConsent"),
                (json_reviews, "verifiedPurchase")),
     columns_of((xml_headers, "Expedited_Delivery"), (xml_deliveries, "On_Time_In_Full"),
                (xml_deliveries, "Signature_Required"), (xml_products, "Recyclable_Packaging"),
                (xml_products, "Active_Flag"), (xml_reviews, "Verified_Purchase")),
     "True / False"),
    ("currency",
     "orders.order_price | orders.delivery_charges | orders.tax_amount | orders.order_total | "
     "order_items.unit_price | order_items.line_revenue | deliveries.delivery_cost | "
     "products.unit_price, products.unit_cost (XML only)",
     columns_of((json_headers, "orderPrice"), (json_headers, "deliveryCharges"),
                (json_headers, "taxAmount"), (json_headers, "orderTotal"),
                (json_items, "unitPrice"), (json_items, "lineRevenue"),
                (json_deliveries, "deliveryCost")),
     columns_of((xml_headers, "Order_Price"), (xml_headers, "Delivery_Charges"),
                (xml_headers, "Tax_Amount"), (xml_headers, "Order_Total"),
                (xml_items, "Unit_Price"), (xml_items, "Line_Revenue"),
                (xml_deliveries, "Delivery_Cost"), (xml_products, "Unit_Price"),
                (xml_products, "Unit_Cost")),
     "float, no label, no thousands separator"),
    ("percentage", "orders.coupon_discount",
     columns_of((json_headers, "couponDiscount")),
     columns_of((xml_headers, "Coupon_Discount")),
     "numeric percentage points (15 not 0.15)"),
    ("missing string", "orders.coupon_code",
     columns_of((json_headers, "couponCode")),
     columns_of((xml_headers, "Coupon_Code")),
     f"literal {MISSING!r}"),
    ("identifier",
     "orders.order_id | order_items.order_item_id | deliveries.delivery_id | "
     "product_reviews.review_id | customers.customer_id (JSON only) | products.product_id (XML only)",
     columns_of((json_headers, "orderID"), (json_items, "orderItemID"),
                (json_deliveries, "deliveryID"), (json_reviews, "reviewID"),
                (json_customers, "customerID")),
     columns_of((xml_headers, "Order_ID"), (xml_items, "Order_Item_ID"),
                (xml_deliveries, "Delivery_ID"), (xml_reviews, "Review_ID"),
                (xml_products, "Product_ID")),
     "verbatim string, leading zeros and case preserved"),
    ("geo / measure",
     "orders.customer_lat | orders.customer_long | deliveries.shipping_distance_km | "
     "deliveries.estimated_carbon_kg | products.weight_kg (XML only)",
     columns_of((json_headers, "customerLat"), (json_headers, "customerLong"),
                (json_deliveries, "shippingDistanceKm"), (json_deliveries, "estimatedCarbonKg")),
     columns_of((xml_headers, "Customer_Lat"), (xml_headers, "Customer_Long"),
                (xml_deliveries, "Shipping_Distance_Km"),
                (xml_deliveries, "Estimated_Carbon_Kg"), (xml_products, "Weight_Kg")),
     "float, published precision preserved"),
]

format_profile = pd.DataFrame([
    {"convention": name, "target_fields_probed": fields,
     "json_values_probed": len(js), "json_representation": pattern_counts(js),
     "xml_values_probed": len(xs), "xml_representation": pattern_counts(xs),
     "target_rule": rule}
    for name, fields, js, xs, rule in conventions
])
save_profile(format_profile, "format_conventions")
format_profile

[saved] profiling/Group005_T1_format_conventions.csv  (8 rows)


,convention,target_fields_probed,json_values_probed,json_representation,xml_values_probed,xml_representation,target_rule
0,date,customers.signup_date | deliveries.dispatch_date | deliveries.promised_date | deliveri...,8954,9999-99-99 (8954),9454,99/99/9999 (9454),YYYY-MM-DD
1,timestamp,orders.order_timestamp | product_reviews.review_timestamp,6764,9999-99-99 99:99:99 (6764),6764,99/99/9999 99:99:99 (6764),YYYY-MM-DD HH:MM:SS
2,boolean,orders.expedited_delivery | deliveries.on_time_in_full | deliveries.signature_required...,12900,True (9233) | False (3667),14400,Y (10538) | N (3862),True / False
3,currency,orders.order_price | orders.delivery_charges | orders.tax_amount | orders.order_total ...,31984,999.99 (12837) | 9999.99 (8961) | 99.99 (5105) | 9.99 (1776),33696,"AUD 999.99 (15336) | AUD 9,999.99 (10691) | AUD 99.99 (5718) | AUD 9.99 (1865)","float, no label, no thousands separator"
4,percentage,orders.coupon_discount,2818,99 (1682) | 9 (1136),2818,99% (1649) | 9% (1169),numeric percentage points (15 not 0.15)
5,missing string,orders.coupon_code,2818,<missing> (1741) | B9SAVE-99 (1077),2818,<missing> (1792) | B9SAVE-99 (1026),literal 'NaN'
6,identifier,orders.order_id | order_items.order_item_id | deliveries.delivery_id | product_reviews...,19029,HITM9999999 (8947) | HREV999999 (3946) | HORD999999 (2818) | HDEL999999 (2818),19385,HITM9999999 (8803) | HREV999999 (3946) | HORD999999 (2818) | HDEL999999 (2818),"verbatim string, leading zeros and case preserved"
7,geo / measure,orders.customer_lat | orders.customer_long | deliveries.shipping_distance_km | deliver...,11272,9.999 (2822) | 999.999999 (2527) | -99.999999 (2509) | 9.9999 (2497),12272,9.999 (3746) | 999.999999 (2516) | 9.9999 (2491) | -99.999999 (2482),"float, published precision preserved"


In [14]:
# Day-first vs month-first is not assumed: the XML day component is checked with pandas .str
xml_date_text = pd.concat([xml_deliveries["Dispatch_Date"], xml_products["Launch_Date"]],
                          ignore_index=True)
print(f"XML date first component max  = {xml_date_text.str.slice(0, 2).astype(int).max()}  -> day  (exceeds 12)")
print(f"XML date second component max = {xml_date_text.str.slice(3, 5).astype(int).max()} -> month")

# Missing-value encodings actually observed, by source. The two exports encode the same
# absence differently, and pandas surfaces that difference directly: JSON carries an empty
# string, while read_xml renders the empty <Coupon_Code/> element as a missing value.
print(f"\nJSON coupon codes absent (empty string \"\")   : {int((json_headers['couponCode'] == '').sum())}")
print(f"JSON coupon codes read as NaN                : {int(json_headers['couponCode'].isna().sum())}")
print(f"XML  coupon codes absent (<Coupon_Code/>)    : {int(xml_headers['Coupon_Code'].isna().sum())}")
print(f"XML  coupon codes that are an empty string   : {int((xml_headers['Coupon_Code'] == '').sum())}")

XML date first component max  = 31  -> day  (exceeds 12)
XML date second component max = 12 -> month

JSON coupon codes absent (empty string "")   : 1741
JSON coupon codes read as NaN                : 0
XML  coupon codes absent (<Coupon_Code/>)    : 1792
XML  coupon codes that are an empty string   : 0


**Material format findings**

| Convention | JSON export | XML export | Target rule |
|---|---|---|---|
| Date | already `YYYY-MM-DD` | `DD/MM/YYYY` | `YYYY-MM-DD` |
| Timestamp | already `YYYY-MM-DD HH:MM:SS` | `DD/MM/YYYY HH:MM:SS` | `YYYY-MM-DD HH:MM:SS` |
| Boolean | native JSON `true`/`false` | `Y` / `N` | `True` / `False` |
| Money | native float | `"AUD 1,234.56"` | float, label and separators removed |
| Percentage | native integer `15` | `"15%"` | numeric percentage **points** |
| Missing string | `""` (empty string) | empty element `<Coupon_Code/>` | literal `NaN` |

* Day-first is **confirmed, not assumed**: the XML first date component reaches 31.
* `coupon_discount` is percentage points; treating `15%` as `0.15` would break `order_total`
  on every discounted order.
* `deliveries.delay_reason` publishes the literal token `none`. That is a **category**, not a
  missing value, and must not be converted to the `NaN` sentinel.

##### Allowed categorical values

The permitted value set of every categorical target field is **measured**, not asserted. The
mapping renders its "Observed domain {...}" phrases from this table at build time, so a domain
claim cannot drift from the data. This is also the evidence a Task 4 check on allowed
categorical values will assert against.

In [15]:
def distinct_values(json_spec, xml_spec):
    '''Distinct non-missing values of a target field across whichever sources carry it.'''
    values = set()
    for spec in (json_spec, xml_spec):
        if spec is None:
            continue
        frame, column = spec
        series = frame[column].dropna().astype(str)
        values |= set(series[series != ""])
    return values

def ordered_domain(values):
    '''Numeric order where the labels are numeric, alphabetical otherwise.'''
    try:
        return [str(v) for v in sorted(values, key=float)]
    except ValueError:
        return sorted(values)

CATEGORICAL_TARGETS = [
    ("orders", "sales_channel",       (json_headers, "salesChannel"),     (xml_headers, "Sales_Channel")),
    ("orders", "payment_method",      (json_headers, "paymentMethod"),    (xml_headers, "Payment_Method")),
    ("orders", "currency",            (json_headers, "currency"),         (xml_headers, "Currency")),
    ("orders", "nearest_warehouse",   (json_headers, "nearestWarehouse"), (xml_headers, "Nearest_Warehouse")),
    ("orders", "order_status",        (json_headers, "orderStatus"),      (xml_headers, "Order_Status")),
    ("orders", "season",              (json_headers, "season"),           (xml_headers, "Season")),
    ("orders", "device_type",         (json_headers, "deviceType"),       (xml_headers, "Device_Type")),
    ("orders", "referral_source",     (json_headers, "referralSource"),   (xml_headers, "Referral_Source")),
    ("order_items", "quantity",       (json_items, "quantity"),           (xml_items, "Quantity")),
    ("customers", "loyalty_tier",     (json_customers, "loyaltyTier"),           None),
    ("customers", "customer_segment", (json_customers, "customerSegment"),       None),
    ("customers", "age_band",         (json_customers, "ageBand"),               None),
    ("customers", "preferred_channel",(json_customers, "preferredChannel"),      None),
    ("customers", "home_state",       (json_customers, "homeState"),             None),
    ("customers", "home_country",     (json_customers, "homeCountry"),           None),
    ("customers", "acquisition_source", (json_customers, "acquisitionSource"),   None),
    ("customers", "account_status",   (json_customers, "accountStatus"),         None),
    ("customers", "preferred_device", (json_customers, "preferredDevice"),       None),
    ("customers", "email_domain",     (json_customers, "emailDomain"),           None),
    ("customers", "household_size_band", (json_customers, "householdSizeBand"),  None),
    ("customers", "contact_frequency_preference", (json_customers, "contactFrequencyPreference"), None),
    ("deliveries", "carrier",         (json_deliveries, "carrier"),       (xml_deliveries, "Carrier")),
    ("deliveries", "service_level",   (json_deliveries, "serviceLevel"),  (xml_deliveries, "Service_Level")),
    ("deliveries", "delivery_status", (json_deliveries, "deliveryStatus"),(xml_deliveries, "Delivery_Status")),
    ("deliveries", "delay_reason",    (json_deliveries, "delayReason"),   (xml_deliveries, "Delay_Reason")),
    ("deliveries", "promised_days",   (json_deliveries, "promisedDays"),  (xml_deliveries, "Promised_Days")),
    ("deliveries", "delivery_window", (json_deliveries, "deliveryWindow"),(xml_deliveries, "Delivery_Window")),
    ("deliveries", "delivery_note_clean", (json_deliveries, "deliveryNoteClean"), (xml_deliveries, "Delivery_Note_Clean")),
    ("products", "brand",             None, (xml_products, "Brand")),
    ("products", "category",          None, (xml_products, "Category")),
    ("products", "subcategory",       None, (xml_products, "Subcategory")),
    ("products", "colour",            None, (xml_products, "Colour")),
    ("products", "supplier_country",  None, (xml_products, "Supplier_Country")),
    ("products", "warranty_months",   None, (xml_products, "Warranty_Months")),
    ("products", "tax_category",      None, (xml_products, "Tax_Category")),
    ("products", "package_type",      None, (xml_products, "Package_Type")),
    ("product_reviews", "language_code",       (json_reviews, "languageCode"),       (xml_reviews, "Language_Code")),
    ("product_reviews", "rating",              (json_reviews, "rating"),             (xml_reviews, "Rating")),
    ("product_reviews", "delivery_experience", (json_reviews, "deliveryExperience"), (xml_reviews, "Delivery_Experience")),
    ("product_reviews", "value_experience",    (json_reviews, "valueExperience"),    (xml_reviews, "Value_Experience")),
    ("product_reviews", "writing_style",       (json_reviews, "writingStyle"),       (xml_reviews, "Writing_Style")),
]

CATEGORICAL_DOMAINS = {(table, field): ordered_domain(distinct_values(js, xs))
                       for table, field, js, xs in CATEGORICAL_TARGETS}

def DOM(table, field):
    '''Render a measured domain phrase for the source-to-target mapping.'''
    values = CATEGORICAL_DOMAINS[(table, field)]
    if len(values) == 1:
        return f" Single observed value {values[0]}."
    return f" Observed domain {{{', '.join(values)}}}."

categorical_profile = pd.DataFrame([
    {"target_table": table, "target_field": field,
     "distinct_values": len(CATEGORICAL_DOMAINS[(table, field)]),
     "observed_domain": ", ".join(CATEGORICAL_DOMAINS[(table, field)])}
    for table, field, _, _ in CATEGORICAL_TARGETS
])
save_profile(categorical_profile, "categorical_domain_profile")
categorical_profile

[saved] profiling/Group005_T1_categorical_domain_profile.csv  (41 rows)


,target_table,target_field,distinct_values,observed_domain
0,orders,sales_channel,3,"Mobile, Store, Web"
1,orders,payment_method,4,"Bank Transfer, Card, Gift Card, PayPal"
2,orders,currency,1,AUD
3,orders,nearest_warehouse,3,"Bakers, Nickolson, Thompson"
4,orders,order_status,1,Completed
5,orders,season,4,"Autumn, Spring, Summer, Winter"
6,orders,device_type,3,"Desktop, Mobile, Tablet"
7,orders,referral_source,5,"Organic, Paid Search, Referral, Social, Store"
8,order_items,quantity,3,"1, 2, 3"
9,customers,loyalty_tier,4,"Bronze, Gold, Platinum, Silver"


##### The published order arithmetic, and how the source rounds

All six published steps are checked on the raw parsed values, before any transformation.

One result deserves its own note. `Series.round(2)` (numpy) and Python's built-in `round`
**disagree** on values landing exactly on a half-cent, because numpy scales by 100 and rounds
the binary result while Python uses a correctly-rounded decimal algorithm. The published
`order_total` figures follow Python's `round`, so that is the rounding used here and in Task 2
(`ASM-14`).

In [16]:
def money_series(series):
    '''Currency column -> float. Handles native floats and 'AUD 1,234.56' text.

    No regex: the currency label is split off as a whitespace token and the
    thousands separators are removed with a literal replace.
    '''
    if pd.api.types.is_numeric_dtype(series):
        return series.astype(float)
    return (series.str.split().str[-1]                       # drop the 'AUD' label
                  .str.replace(",", "", regex=False)          # drop thousands separators
                  .astype(float))

def percent_series(series):
    if pd.api.types.is_numeric_dtype(series):
        return series.astype(float)
    return series.str.rstrip("%").astype(float)

def round2(series):
    '''Round to cents the way the source did: Python's round, not numpy's.'''
    return series.map(lambda value: round(float(value), 2))

# Evidence for ASM-14: how far the two rounding rules disagree on this package.
_price = money_series(json_headers["orderPrice"])
_disc = percent_series(json_headers["couponDiscount"])
_charges = money_series(json_headers["deliveryCharges"])
_published = money_series(json_headers["orderTotal"])
_raw = _price * (1 - _disc / 100) + _charges
rounding_evidence = pd.DataFrame([
    {"rounding_rule": "pandas/numpy Series.round(2)",
     "orders_matching_published_total": int((_raw.round(2).sub(_published).abs() <= MONEY_TOLERANCE).sum())},
    {"rounding_rule": "Python round(value, 2)",
     "orders_matching_published_total": int((round2(_raw).sub(_published).abs() <= MONEY_TOLERANCE).sum())},
]).assign(orders_probed=len(json_headers))
rounding_evidence

,rounding_rule,orders_matching_published_total,orders_probed
0,pandas/numpy Series.round(2),2794,2818
1,"Python round(value, 2)",2818,2818


In [17]:
# Steps 1 and 2, on the DE-DUPLICATED records so the check matches the T1-P6 reconciliation.
# The final column shows what happens if the within-source repeats are not removed first.
def check_steps_1_and_2(label, items, headers, item_cols, header_cols):
    item_key, order_col, qty_col, price_col, line_col = item_cols
    order_key, order_price_col = header_cols

    lines = pd.DataFrame({
        "order_item_id": items[item_key],
        "order_id": items[order_col],
        "derived": round2(items[qty_col].astype(float) * money_series(items[price_col])),
        "published": round2(money_series(items[line_col])),
    })
    distinct_lines = lines.drop_duplicates(subset="order_item_id")
    step1_ok = int((distinct_lines["derived"].sub(distinct_lines["published"]).abs()
                    <= MONEY_TOLERANCE).sum())

    published_price = (pd.DataFrame({"order_id": headers[order_key],
                                     "order_price": round2(money_series(headers[order_price_col]))})
                       .drop_duplicates(subset="order_id").set_index("order_id")["order_price"])
    dedup = round2(distinct_lines.groupby("order_id")["derived"].sum())
    with_repeats = round2(lines.groupby("order_id")["derived"].sum())

    return {
        "source": label,
        "distinct_items": len(distinct_lines),
        "step 1: line_revenue == round(qty*unit_price, 2)": f"{step1_ok}/{len(distinct_lines)}",
        "distinct_orders": len(published_price),
        "step 2: order_price == sum(line_revenue), de-duplicated":
            f"{int((dedup.reindex(published_price.index).sub(published_price).abs() <= MONEY_TOLERANCE).sum())}/{len(published_price)}",
        "step 2 if within-source repeats were kept":
            f"{int((with_repeats.reindex(published_price.index).sub(published_price).abs() <= MONEY_TOLERANCE).sum())}/{len(published_price)}",
    }

steps_12 = pd.DataFrame([
    check_steps_1_and_2("JSON", json_items, json_headers,
                        ("orderItemID", "orderID", "quantity", "unitPrice", "lineRevenue"),
                        ("orderID", "orderPrice")),
    check_steps_1_and_2("XML", xml_items, xml_headers,
                        ("Order_Item_ID", "Order_ID", "Quantity", "Unit_Price", "Line_Revenue"),
                        ("Order_ID", "Order_Price")),
])

# Steps 3 to 6, on the order header rows as published.
arithmetic_rows = []
for label, headers, cols in [
    ("JSON", json_headers, ("orderPrice", "couponDiscount", "deliveryCharges", "taxAmount", "orderTotal")),
    ("XML", xml_headers, ("Order_Price", "Coupon_Discount", "Delivery_Charges", "Tax_Amount", "Order_Total")),
]:
    price = money_series(headers[cols[0]])
    discount = percent_series(headers[cols[1]])
    charges = money_series(headers[cols[2]])
    tax = round2(money_series(headers[cols[3]]))
    total = round2(money_series(headers[cols[4]]))
    arithmetic_rows.append({
        "source": label,
        "order_rows_probed": len(headers),
        "step 3: tax == round(order_price/11, 2)":
            f"{int((round2(price / 11).sub(tax).abs() <= MONEY_TOLERANCE).sum())}/{len(headers)}",
        "steps 4-6: total == round(price*(1-disc/100)+charges, 2)":
            f"{int((round2(price * (1 - discount / 100) + charges).sub(total).abs() <= MONEY_TOLERANCE).sum())}/{len(headers)}",
    })
arithmetic_steps = pd.DataFrame(arithmetic_rows)
save_profile(steps_12.merge(arithmetic_steps, on="source"), "order_arithmetic_checks")
print(steps_12)
arithmetic_steps

[saved] profiling/Group005_T1_order_arithmetic_checks.csv  (2 rows)
  source  distinct_items step 1: line_revenue == round(qty*unit_price, 2)  distinct_orders step 2: order_price == sum(line_revenue), de-duplicated step 2 if within-source repeats were kept
0   JSON            8723                                        8723/8723             2750                                               2750/2750                                 2682/2750
1    XML            8618                                        8618/8618             2750                                               2750/2750                                 2682/2750


,source,order_rows_probed,"step 3: tax == round(order_price/11, 2)","steps 4-6: total == round(price*(1-disc/100)+charges, 2)"
0,JSON,2818,2818/2818,2818/2818
1,XML,2818,2818/2818,2818/2818


---
#### `T1-P5` — Field coverage: which target fields come from one source or both

In [18]:
dictionary = pd.read_csv(DICTIONARY_PATH, keep_default_na=False)
print(f"required target fields: {len(dictionary)}")
dictionary.groupby(["output_table", "grain"], sort=False).size().rename("fields").reset_index()

required target fields: 111


,output_table,grain,fields
0,orders,one row per order,23
1,order_items,one row per order item,6
2,customers,one row per customer,20
3,deliveries,one row per completed order,20
4,products,one row per product,21
5,product_reviews,one row per canonical review,21


In [19]:
# Source elements that are read but deliberately carry no target field.
unmapped = pd.DataFrame([
    ("JSON", "$.exportMetadata", "group alias, period, source system",
     "Used to confirm the package identity in T1-P1; not a business entity in the dictionary."),
    ("XML", "/OperationsExport/Export_Metadata", "system name, contract version",
     "Used to confirm the package identity in T1-P1; not a business entity in the dictionary."),
    ("XML", "/OperationsExport/WarehouseDirectory/Warehouse", "warehouse name, latitude, longitude",
     "No warehouse table is required. Used only to cross-check the orders.nearest_warehouse vocabulary."),
], columns=["source", "structural_path", "content", "treatment"])

warehouse_names = set(xml_warehouses["Name"])
order_warehouses = set(json_headers["nearestWarehouse"]) | set(xml_headers["Nearest_Warehouse"])
print("WarehouseDirectory names      :", sorted(warehouse_names))
print("nearest_warehouse values used :", sorted(order_warehouses))
print("vocabulary matches            :", warehouse_names == order_warehouses)
unmapped

WarehouseDirectory names      : ['Bakers', 'Nickolson', 'Thompson']
nearest_warehouse values used : ['Bakers', 'Nickolson', 'Thompson']
vocabulary matches            : True


,source,structural_path,content,treatment
0,JSON,$.exportMetadata,"group alias, period, source system",Used to confirm the package identity in T1-P1; not a business entity in the dictionary.
1,XML,/OperationsExport/Export_Metadata,"system name, contract version",Used to confirm the package identity in T1-P1; not a business entity in the dictionary.
2,XML,/OperationsExport/WarehouseDirectory/Warehouse,"warehouse name, latitude, longitude",No warehouse table is required. Used only to cross-check the orders.nearest_warehouse ...


---
#### `T1-P6` — Duplicate records within a source and overlapping records across sources

Overlap cannot be judged on raw values, because the same order is written as `AUD 5,287.78` /
`14/10/2018 09:28:00` / `Y` in one system and `5287.78` / `2018-10-14 09:28:00` / `true` in
the other. Comparable columns are therefore **normalised first**, then records are compared
field-by-field on the table primary key.

The normalisers below are profiling-scope; Task 2 reuses the same conventions.

In [20]:
# --- vectorised, source-aware normalisers ----------------------------------
def is_absent(value):
    '''One definition of "missing" for all three source encodings.

    JSON writes an empty string, read_xml renders an empty element as NaN, and a
    genuinely absent key is None. Collapsing them here is what makes ASM-04 true and
    stops the same absence being read as a cross-source conflict.
    '''
    return value is None or value == "" or (isinstance(value, float) and pd.isna(value))

def norm_id(series):
    return series.astype(object).map(lambda v: None if is_absent(v) else str(v).strip())

def norm_text(series):
    def clean(value):
        if is_absent(value):
            return None
        # str.split() collapses any run of whitespace and trims - no regex needed here.
        return " ".join(html.unescape(unicodedata.normalize("NFC", str(value))).split())
    return series.map(clean)

def norm_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)
    return series.map({"Y": True, "N": False})

def norm_int(series):
    return series.astype(float).astype(int)

def norm_float(series, places=6):
    return series.astype(float).round(places)

def norm_date(series, source):
    fmt = "%Y-%m-%d" if source == "JSON" else "%d/%m/%Y"
    return pd.to_datetime(series, format=fmt).dt.strftime("%Y-%m-%d")

def norm_datetime(series, source):
    fmt = "%Y-%m-%d %H:%M:%S" if source == "JSON" else "%d/%m/%Y %H:%M:%S"
    return pd.to_datetime(series, format=fmt).dt.strftime("%Y-%m-%d %H:%M:%S")

# Column name per source, so one normalisation spec serves both exports.
COLUMN_MAP = {
    "orders": {
        "JSON": dict(order_id="orderID", source_system_record_id="sourceSystemRecordID",
                     customer_id="customerID", order_timestamp="orderTimestamp",
                     sales_channel="salesChannel", payment_method="paymentMethod", currency="currency",
                     nearest_warehouse="nearestWarehouse", order_status="orderStatus",
                     order_price="orderPrice", delivery_charges="deliveryCharges",
                     coupon_code="couponCode", coupon_discount="couponDiscount",
                     tax_amount="taxAmount", order_total="orderTotal", season="season",
                     expedited_delivery="expeditedDelivery", customer_lat="customerLat",
                     customer_long="customerLong", device_type="deviceType",
                     referral_source="referralSource", customer_note="customerNote"),
        "XML": dict(order_id="Order_ID", source_system_record_id="Source_System_Record_ID",
                    customer_id="Customer_ID", order_timestamp="Order_Timestamp",
                    sales_channel="Sales_Channel", payment_method="Payment_Method", currency="Currency",
                    nearest_warehouse="Nearest_Warehouse", order_status="Order_Status",
                    order_price="Order_Price", delivery_charges="Delivery_Charges",
                    coupon_code="Coupon_Code", coupon_discount="Coupon_Discount",
                    tax_amount="Tax_Amount", order_total="Order_Total", season="Season",
                    expedited_delivery="Expedited_Delivery", customer_lat="Customer_Lat",
                    customer_long="Customer_Long", device_type="Device_Type",
                    referral_source="Referral_Source", customer_note="Customer_Note"),
    },
    "order_items": {
        "JSON": dict(order_item_id="orderItemID", order_id="orderID", product_id="productID",
                     quantity="quantity", unit_price="unitPrice", line_revenue="lineRevenue"),
        "XML": dict(order_item_id="Order_Item_ID", order_id="Order_ID", product_id="Product_ID",
                    quantity="Quantity", unit_price="Unit_Price", line_revenue="Line_Revenue"),
    },
    "deliveries": {
        "JSON": dict(delivery_id="deliveryID", order_id="orderID", dispatch_date="dispatchDate",
                     promised_date="promisedDate", delivered_date="deliveredDate", carrier="carrier",
                     service_level="serviceLevel", delivery_status="deliveryStatus",
                     delay_days="delayDays", on_time_in_full="onTimeInFull",
                     fulfilment_hours="fulfilmentHours", delivery_cost="deliveryCost",
                     delay_reason="delayReason", promised_days="promisedDays",
                     tracking_event_count="trackingEventCount", delivery_window="deliveryWindow",
                     shipping_distance_km="shippingDistanceKm", signature_required="signatureRequired",
                     estimated_carbon_kg="estimatedCarbonKg", delivery_note_clean="deliveryNoteClean"),
        "XML": dict(delivery_id="Delivery_ID", order_id="Order_ID", dispatch_date="Dispatch_Date",
                    promised_date="Promised_Date", delivered_date="Delivered_Date", carrier="Carrier",
                    service_level="Service_Level", delivery_status="Delivery_Status",
                    delay_days="Delay_Days", on_time_in_full="On_Time_In_Full",
                    fulfilment_hours="Fulfilment_Hours", delivery_cost="Delivery_Cost",
                    delay_reason="Delay_Reason", promised_days="Promised_Days",
                    tracking_event_count="Tracking_Event_Count", delivery_window="Delivery_Window",
                    shipping_distance_km="Shipping_Distance_Km", signature_required="Signature_Required",
                    estimated_carbon_kg="Estimated_Carbon_Kg", delivery_note_clean="Delivery_Note_Clean"),
    },
    "product_reviews": {
        "JSON": dict(review_id="reviewID", order_id="orderID", order_item_id="orderItemID",
                     product_id="productID", customer_id="customerID",
                     review_timestamp="reviewTimestamp", language_code="languageCode", rating="rating",
                     review_title="reviewTitle", verified_purchase="verifiedPurchase",
                     helpful_votes="helpfulVotes", delivery_experience="deliveryExperience",
                     value_experience="valueExperience", writing_style="writingStyle",
                     review_text="reviewText"),
        "XML": dict(review_id="Review_ID", order_id="Order_ID", order_item_id="Order_Item_ID",
                    product_id="Product_ID", customer_id="Customer_ID",
                    review_timestamp="Review_Timestamp", language_code="Language_Code", rating="Rating",
                    review_title="Review_Title", verified_purchase="Verified_Purchase",
                    helpful_votes="Helpful_Votes", delivery_experience="Delivery_Experience",
                    value_experience="Value_Experience", writing_style="Writing_Style",
                    review_text="Review_Text"),
    },
}

# How each normalised column is produced, independent of source spelling.
FIELD_RULES = {
    "orders": {
        "order_id": norm_id, "source_system_record_id": norm_id, "customer_id": norm_id,
        "order_timestamp": "datetime", "sales_channel": norm_id, "payment_method": norm_id,
        "currency": norm_id, "nearest_warehouse": norm_id, "order_status": norm_id,
        "order_price": "money", "delivery_charges": "money", "coupon_code": norm_id,
        "coupon_discount": "percent", "tax_amount": "money", "order_total": "money",
        "season": norm_id, "expedited_delivery": norm_bool, "customer_lat": "float6",
        "customer_long": "float6", "device_type": norm_id, "referral_source": norm_id,
        "customer_note": norm_text,
    },
    "order_items": {
        "order_item_id": norm_id, "order_id": norm_id, "product_id": norm_id,
        "quantity": norm_int, "unit_price": "money", "line_revenue": "money",
    },
    "deliveries": {
        "delivery_id": norm_id, "order_id": norm_id, "dispatch_date": "date",
        "promised_date": "date", "delivered_date": "date", "carrier": norm_id,
        "service_level": norm_id, "delivery_status": norm_id, "delay_days": norm_int,
        "on_time_in_full": norm_bool, "fulfilment_hours": norm_int, "delivery_cost": "money",
        "delay_reason": norm_id, "promised_days": norm_int, "tracking_event_count": norm_int,
        "delivery_window": norm_id, "shipping_distance_km": "float4",
        "signature_required": norm_bool, "estimated_carbon_kg": "float3",
        "delivery_note_clean": norm_text,
    },
    "product_reviews": {
        "review_id": norm_id, "order_id": norm_id, "order_item_id": norm_id, "product_id": norm_id,
        "customer_id": norm_id, "review_timestamp": "datetime", "language_code": norm_id,
        "rating": norm_int, "review_title": norm_text, "verified_purchase": norm_bool,
        "helpful_votes": norm_int, "delivery_experience": norm_id, "value_experience": norm_id,
        "writing_style": norm_id, "review_text": norm_text,
    },
}

def normalise(frame, table, source):
    '''Return a DataFrame with target column names and comparable, source-neutral values.'''
    columns, rules = COLUMN_MAP[table][source], FIELD_RULES[table]
    out = {}
    for target, rule in rules.items():
        series = frame[columns[target]]
        if rule == "money":       out[target] = round2(money_series(series))
        elif rule == "percent":   out[target] = percent_series(series)
        elif rule == "date":      out[target] = norm_date(series, source)
        elif rule == "datetime":  out[target] = norm_datetime(series, source)
        elif rule == "float6":    out[target] = norm_float(series, 6)
        elif rule == "float4":    out[target] = norm_float(series, 4)
        elif rule == "float3":    out[target] = norm_float(series, 3)
        else:                     out[target] = rule(series)
    return pd.DataFrame(out)

NORMALISED = {
    ("orders", "JSON"): normalise(json_headers, "orders", "JSON"),
    ("orders", "XML"): normalise(xml_headers, "orders", "XML"),
    ("order_items", "JSON"): normalise(json_items, "order_items", "JSON"),
    ("order_items", "XML"): normalise(xml_items, "order_items", "XML"),
    ("deliveries", "JSON"): normalise(json_deliveries, "deliveries", "JSON"),
    ("deliveries", "XML"): normalise(xml_deliveries, "deliveries", "XML"),
    ("product_reviews", "JSON"): normalise(json_reviews, "product_reviews", "JSON"),
    ("product_reviews", "XML"): normalise(xml_reviews, "product_reviews", "XML"),
    # Single-source tables need no cross-source comparison, only their own key check.
    ("customers", "JSON"): json_customers.rename(columns={"customerID": "customer_id"}),
    ("products", "XML"): xml_products.rename(columns={"Product_ID": "product_id"}),
}

PRIMARY_KEYS = {"orders": "order_id", "order_items": "order_item_id", "deliveries": "delivery_id",
                "product_reviews": "review_id", "customers": "customer_id", "products": "product_id"}

for (table, source), frame in NORMALISED.items():
    print(f"  {table:16s} {source:4s} {frame.shape[0]:6d} rows x {frame.shape[1]:2d} normalised columns")

  orders           JSON   2818 rows x 22 normalised columns
  orders           XML    2818 rows x 22 normalised columns
  order_items      JSON   8947 rows x  6 normalised columns
  order_items      XML    8803 rows x  6 normalised columns
  deliveries       JSON   2818 rows x 20 normalised columns
  deliveries       XML    2818 rows x 20 normalised columns
  product_reviews  JSON   3946 rows x 15 normalised columns
  product_reviews  XML    3946 rows x 15 normalised columns
  customers        JSON    500 rows x 20 normalised columns
  products         XML    1000 rows x 21 normalised columns


In [21]:
conflict_log = []   # every field-level disagreement is recorded, never silently resolved

def within_source_repeats(frame, key, scope):
    '''Split repeated key rows into identical repeats and genuine conflicts.'''
    if frame.empty:
        return 0, 0
    comparable = frame.astype(str)                      # NaN-safe row comparison
    repeat_rows = len(frame) - frame[key].nunique()
    identical_rows = len(frame) - len(comparable.drop_duplicates())
    conflicting = repeat_rows - identical_rows
    if conflicting:
        duplicated_keys = frame[key][frame[key].duplicated(keep=False)]
        for k, group in comparable[frame[key].isin(duplicated_keys)].groupby(key):
            first = group.iloc[0]
            for _, other in group.iloc[1:].iterrows():
                differing = [c for c in group.columns if first[c] != other[c]]
                if differing:
                    conflict_log.append({"scope": scope, "key_name": key, "key": k,
                                         "fields": ", ".join(differing)})
    return repeat_rows, conflicting

def cross_source_conflicts(json_frame, xml_frame, key, scope):
    '''Compare the two exports field-by-field on their shared keys.'''
    if json_frame.empty or xml_frame.empty:
        return 0, 0
    left = json_frame.drop_duplicates(subset=key).set_index(key).astype(str)
    right = xml_frame.drop_duplicates(subset=key).set_index(key).astype(str)
    shared = left.index.intersection(right.index)
    if shared.empty:
        return 0, 0
    left, right = left.loc[shared], right.loc[shared, left.columns]
    differs = left.ne(right)                            # vectorised cell-wise comparison
    rows_with_conflict = differs.any(axis=1)
    for k in differs.index[rows_with_conflict]:
        conflict_log.append({"scope": scope, "key_name": key, "key": k,
                             "fields": ", ".join(differs.columns[differs.loc[k]])})
    return len(shared), int(rows_with_conflict.sum())

overlap_rows = []
for table, key in PRIMARY_KEYS.items():
    json_frame = NORMALISED.get((table, "JSON"), pd.DataFrame())
    xml_frame = NORMALISED.get((table, "XML"), pd.DataFrame())

    json_repeats, json_conflicting = within_source_repeats(json_frame, key, f"{table}: within JSON")
    xml_repeats, xml_conflicting = within_source_repeats(xml_frame, key, f"{table}: within XML")
    shared, cross_conflicting = cross_source_conflicts(json_frame, xml_frame, key,
                                                       f"{table}: JSON vs XML")
    json_keys = set(json_frame[key]) if not json_frame.empty else set()
    xml_keys = set(xml_frame[key]) if not xml_frame.empty else set()

    overlap_rows.append({
        "target_table": table, "primary_key": key,
        "json_records": len(json_frame), "json_distinct_keys": len(json_keys),
        "xml_records": len(xml_frame), "xml_distinct_keys": len(xml_keys),
        "within_json_repeat_pairs": json_repeats, "within_json_conflicting": json_conflicting,
        "within_xml_repeat_pairs": xml_repeats, "within_xml_conflicting": xml_conflicting,
        "keys_in_both_sources": shared, "cross_source_conflicting": cross_conflicting,
        "canonical_keys_expected": len(json_keys | xml_keys),
    })

overlap_profile = pd.DataFrame(overlap_rows)
save_profile(overlap_profile, "duplicate_and_overlap_profile")
overlap_profile

[saved] profiling/Group005_T1_duplicate_and_overlap_profile.csv  (6 rows)


,target_table,primary_key,json_records,json_distinct_keys,xml_records,xml_distinct_keys,within_json_repeat_pairs,within_json_conflicting,within_xml_repeat_pairs,within_xml_conflicting,keys_in_both_sources,cross_source_conflicting,canonical_keys_expected
0,orders,order_id,2818,2750,2818,2750,68,0,68,0,500,0,5000
1,order_items,order_item_id,8947,8723,8803,8618,224,0,185,0,1602,0,15739
2,deliveries,delivery_id,2818,2750,2818,2750,68,0,68,0,500,0,5000
3,product_reviews,review_id,3946,3850,3946,3850,96,0,96,0,700,0,7000
4,customers,customer_id,500,500,0,0,0,0,0,0,0,0,500
5,products,product_id,0,0,1000,1000,0,0,0,0,0,0,1000


In [22]:
conflicts = pd.DataFrame(conflict_log, columns=["scope", "key_name", "key", "fields"])
save_profile(conflicts, "field_level_conflicts")
print(f"field-level conflicts recorded: {len(conflicts)}")
print(conflicts.head(20) if len(conflicts) else "No field-level conflict detected after normalisation.")

[saved] profiling/Group005_T1_field_level_conflicts.csv  (0 rows)
field-level conflicts recorded: 0
No field-level conflict detected after normalisation.


**Material reconciliation findings**

* **Both duplication types are present and must be handled in that order.** Each export
  repeats a subset of its own records verbatim (within-source duplication), *and* the two
  exports independently publish a shared subset of orders, items, deliveries and reviews
  (cross-source overlap). Concatenating first and de-duplicating once on the primary key
  handles both without double counting.
* **After the published normalisation, every repeat agrees on every compared field** — zero
  field-level conflicts within a source and zero across sources. A correctly normalised
  duplicate therefore needs no arbitrary JSON-over-XML precedence rule, and **none is applied**.
* The conflict log is written to `profiling/Group005_T1_field_level_conflicts.csv`. It is
  produced by the same code path that would record a conflict, so the empty file is evidence
  rather than an absence of checking.
* `customers` and `products` are single-source and already unique on their keys, so they need
  no reconciliation at all.

---
#### `T1-P7` — Source-level referential integrity

Foreign keys are checked against the **union** of both sources, because a child row in one
export may legitimately reference a parent that only the other export publishes.

In [23]:
def canonical_keys(table):
    keys = set()
    for source in ("JSON", "XML"):
        frame = NORMALISED.get((table, source))
        if frame is not None and not frame.empty:
            keys |= set(frame[PRIMARY_KEYS[table]])
    return keys

def referenced_values(table, column):
    values = set()
    for source in ("JSON", "XML"):
        frame = NORMALISED.get((table, source))
        if frame is not None and not frame.empty:
            values |= set(frame[column].dropna())
    return values

CANONICAL = {table: canonical_keys(table) for table in PRIMARY_KEYS}

fk_checks = [
    ("orders.customer_id",            "customers.customer_id",     ("orders", "customer_id"),            "customers"),
    ("order_items.order_id",          "orders.order_id",           ("order_items", "order_id"),          "orders"),
    ("order_items.product_id",        "products.product_id",       ("order_items", "product_id"),        "products"),
    ("deliveries.order_id",           "orders.order_id",           ("deliveries", "order_id"),           "orders"),
    ("product_reviews.order_id",      "orders.order_id",           ("product_reviews", "order_id"),      "orders"),
    ("product_reviews.order_item_id", "order_items.order_item_id", ("product_reviews", "order_item_id"), "order_items"),
    ("product_reviews.product_id",    "products.product_id",       ("product_reviews", "product_id"),    "products"),
    ("product_reviews.customer_id",   "customers.customer_id",     ("product_reviews", "customer_id"),   "customers"),
]

fk_profile = pd.DataFrame([
    {
        "child_field": child, "parent_field": parent,
        "distinct_values_referenced": len(child_values),
        "distinct_parent_keys": len(parent_keys),
        "orphans_against_union": len(child_values - parent_keys),
        "parent_coverage_pct": round(100 * len(child_values & parent_keys) / len(parent_keys), 2),
        "status": "PASS" if not (child_values - parent_keys) else "FAIL",
    }
    for child, parent, (child_table, child_column), parent_table in fk_checks
    for child_values in [referenced_values(child_table, child_column)]
    for parent_keys in [CANONICAL[parent_table]]
])
save_profile(fk_profile, "source_referential_integrity")
fk_profile

[saved] profiling/Group005_T1_source_referential_integrity.csv  (8 rows)


,child_field,parent_field,distinct_values_referenced,distinct_parent_keys,orphans_against_union,parent_coverage_pct,status
0,orders.customer_id,customers.customer_id,500,500,0,100.00,PASS
1,order_items.order_id,orders.order_id,5000,5000,0,100.00,PASS
2,order_items.product_id,products.product_id,1000,1000,0,100.00,PASS
3,deliveries.order_id,orders.order_id,5000,5000,0,100.00,PASS
4,product_reviews.order_id,orders.order_id,4017,5000,0,80.34,PASS
5,product_reviews.order_item_id,order_items.order_item_id,7000,15739,0,44.48,PASS
6,product_reviews.product_id,products.product_id,1000,1000,0,100.00,PASS
7,product_reviews.customer_id,customers.customer_id,500,500,0,100.00,PASS


**Material integrity findings**

* All eight required relationships resolve against the union of the two exports, with **zero
  orphans**. Neither source is self-sufficient — the XML order lines reference products the
  JSON export never mentions, and both exports' orders reference customers only the JSON
  export publishes — so reconciliation must complete *before* integrity can be asserted.
* Every customer and every catalogue product is actually referenced by transactional data.

---
#### `T1-P9` — Assumptions register

Assumptions that had to be settled before transformation can begin. Each is stated with the
evidence that supports it and the consequence if it is wrong.

In [24]:
assumptions = pd.DataFrame([
    ("ASM-01", "XML dates are day-first DD/MM/YYYY.",
     "T1-P4: the first component reaches 31, which cannot be a month.",
     "Month/day inversion on every XML date, breaking temporal ordering checks."),
    ("ASM-02", "coupon_discount is percentage points, not a fraction.",
     "T1-P4: XML publishes '15%'; applying 15/100 reproduces order_total on every order in both sources.",
     "order_total would be wrong on every discounted order."),
    ("ASM-03", "Product prices are GST-inclusive; tax_amount is the included component order_price/11 taken before the discount.",
     "T1-P4: round(order_price/11, 2) reproduces the published tax_amount on all orders; products.tax_category is GST_STANDARD.",
     "GST would be added again and order_total overstated."),
    ("ASM-04", "Absence of a coupon code is the same fact in both sources.",
     "T1-P4: JSON writes an empty string, XML writes an empty <Coupon_Code/> element; both mean 'no coupon'.",
     "Two encodings of the same absence would be read as a cross-source conflict."),
    ("ASM-05", "A missing prescribed string is written as the literal three characters NaN, not an empty cell or a pandas null.",
     "Specification s.4 Task 2; applies to coupon_code, promo_code and the derived narrative fields.",
     "Missing values would be unreadable to the marking comparison."),
    ("ASM-06", "deliveries.delay_reason value 'none' is a category, not a missing value.",
     "T1-P4: it co-occurs with delay_days = 0 and on_time_in_full = True on every such row.",
     "A valid category would be destroyed and on-time deliveries would look unexplained."),
    ("ASM-07", "No source precedence rule is applied when reconciling.",
     "T1-P6: zero field-level conflicts within or across sources after normalisation.",
     "An arbitrary precedence rule would hide a genuine conflict if one ever appeared."),
    ("ASM-08", "deliveries is 1:1 with orders in this package because every order is Completed.",
     "T1-P3: Order_Status is 'Completed' and Delivery_Status is 'Delivered' on all rows in both exports.",
     "No filter is needed for this package, but Task 2 must apply an explicit completed-order filter or a package with open orders would silently break the deliveries grain."),
    ("ASM-09", "product_reviews.language_code comes from the structured source attribute only.",
     "T1-P8 / specification s.4 Task 3: language must not be inferred from the Latin-analysis field.",
     "Inferred language would misclassify every accented-Latin and mixed-script review."),
    ("ASM-10", "Review timestamps after 2018-12-31 are valid, not errors.",
     "T1-P4: reviews run to 2019-02-20 while orders stop at 2018-12-31; a review follows its delivery.",
     "Valid late reviews would be dropped as out-of-period."),
    ("ASM-11", "products.unit_price is the current catalogue price and does not overwrite order_items.unit_price.",
     "T1-P3: order lines are historical 2018 events; catalogue prices are a separate current-state attribute.",
     "Historical line revenue and order arithmetic would be retrospectively rewritten."),
    ("ASM-12", "Inactive products (active_flag = N) are retained.",
     "T1-P4: 40 of 1,000 products are inactive yet still appear on 2018 order lines.",
     "Order items would be orphaned from their product parent."),
    ("ASM-13", "When review_body_clean is the literal NaN sentinel, review_length_chars = 0, review_word_count = 0 and contains_non_latin_script = False.",
     "Specification s.4 Task 3 requires the sentinel not to be counted as an ordinary review, and the dictionary types these three fields numeric/numeric/boolean and non-nullable, so the string NaN cannot be written into them. T1-P8: no value in this package cleans to empty, so the rule is defined for the contract and for private tests rather than exercised here.",
     "Length 3 and word count 1 would treat the sentinel as a one-word review and bias every text-length statistic; writing the string NaN would break the required numeric/boolean types."),
    ("ASM-14", "Monetary rounding uses Python's round(value, 2), not pandas/numpy Series.round(2).",
     "T1-P4: the two rules disagree on half-cent values because numpy scales by 100 and rounds the binary result. Python's round reproduces the published order_total on every order in both exports; numpy's does not.",
     "Roughly 1% of orders would fail the published order_total contract at tolerance 0.01, and Task 2 would emit values the marking comparison rejects."),
    ("ASM-15", "The Latin analysis keeps Latin-script letters with their combining marks, keeps digits and narrow punctuation, and drops non-Latin letters together with the wide and fullwidth punctuation that belongs to the same script.",
     "Specification s.4 Task 3 retains 'applicable' digits and punctuation. Wide and fullwidth punctuation (U+3001, U+FF0C, U+3002) is applicable to the script being removed, not to the Latin analysis: keeping it leaves a residue such as 'candle quest 768 , , . .' on a Chinese review. Narrow Latin typography (<<, >>, em dash) is ambiguous or narrow width and survives. NFC composes European diacritics; a combining mark is kept only after a retained Latin base.",
     "Keeping wide punctuation would leave meaningless residue from the removed script; dropping narrow punctuation would discard characters the contract keeps."),
    ("ASM-16", "coupon_discount is written as an integer while every observed discount is a whole percentage point.",
     "The dictionary compares coupon_discount 'exact after published normalisation' rather than on a tolerance, and both exports publish whole points ('15' / '15%'). The cast is conditional on the parsed column being integral, so a fractional discount in another package keeps the float representation.",
     "'15.0' would be the only float stand-in for an integer among the six outputs and could fail an exact string comparison."),
], columns=["assumption_id", "assumption", "evidence", "consequence_if_wrong"])
save_profile(assumptions, "assumptions_register")
assumptions

[saved] profiling/Group005_T1_assumptions_register.csv  (16 rows)


,assumption_id,assumption,evidence,consequence_if_wrong
0,ASM-01,XML dates are day-first DD/MM/YYYY.,"T1-P4: the first component reaches 31, which cannot be a month.","Month/day inversion on every XML date, breaking temporal ordering checks."
1,ASM-02,"coupon_discount is percentage points, not a fraction.",T1-P4: XML publishes '15%'; applying 15/100 reproduces order_total on every order in b...,order_total would be wrong on every discounted order.
2,ASM-03,Product prices are GST-inclusive; tax_amount is the included component order_price/11 ...,"T1-P4: round(order_price/11, 2) reproduces the published tax_amount on all orders; pro...",GST would be added again and order_total overstated.
3,ASM-04,Absence of a coupon code is the same fact in both sources.,"T1-P4: JSON writes an empty string, XML writes an empty <Coupon_Code/> element; both m...",Two encodings of the same absence would be read as a cross-source conflict.
4,ASM-05,"A missing prescribed string is written as the literal three characters NaN, not an emp...","Specification s.4 Task 2; applies to coupon_code, promo_code and the derived narrative...",Missing values would be unreadable to the marking comparison.
5,ASM-06,"deliveries.delay_reason value 'none' is a category, not a missing value.",T1-P4: it co-occurs with delay_days = 0 and on_time_in_full = True on every such row.,A valid category would be destroyed and on-time deliveries would look unexplained.
6,ASM-07,No source precedence rule is applied when reconciling.,T1-P6: zero field-level conflicts within or across sources after normalisation.,An arbitrary precedence rule would hide a genuine conflict if one ever appeared.
7,ASM-08,deliveries is 1:1 with orders in this package because every order is Completed.,T1-P3: Order_Status is 'Completed' and Delivery_Status is 'Delivered' on all rows in b...,"No filter is needed for this package, but Task 2 must apply an explicit completed-orde..."
8,ASM-09,product_reviews.language_code comes from the structured source attribute only.,T1-P8 / specification s.4 Task 3: language must not be inferred from the Latin-analysi...,Inferred language would misclassify every accented-Latin and mixed-script review.
9,ASM-10,"Review timestamps after 2018-12-31 are valid, not errors.",T1-P4: reviews run to 2019-02-20 while orders stop at 2018-12-31; a review follows its...,Valid late reviews would be dropped as out-of-period.


## 2. Source-to-target mapping

---
#### `T1-M1` — Source-to-target mapping

The mapping is a **data artefact, not code**. It lives in
`Group005_source_to_target_mapping.csv` — one of the required submission files — so any group
member can edit it in a spreadsheet without touching Python.

This notebook therefore **audits** the mapping rather than generating it. Six checks run
against the published contract and against the measurements made earlier in this notebook, so
a hand edit that breaks the contract, invents a path, or contradicts the measured data fails
loudly instead of reaching the marker.

| Column | Content |
|---|---|
| `mapping_id` | stable `MAP-###` identifier (Appendix A submission checklist) |
| `source_format` | `JSON`, `XML`, `both` or `derived` |
| `json_source_path` | JSONPath-style structural path, or `N/A` |
| `xml_source_path` | absolute XPath, or `N/A` |
| `transformation_or_derivation` | the normalisation or derivation applied |
| `overlap_or_conflict_rule` | how duplication/overlap and disagreement are handled |
| `notebook_evidence` | stable Task 1 section IDs supporting the row |

Path columns hold **only** structural paths, several inputs separated by `|`. Where a derived
field is built from another *target* field, that dependency is stated in
`transformation_or_derivation`, not smuggled into the path.

In [25]:
# T1-M1: audit the hand-maintained mapping CSV against the contract and the measurements.
mapping = pd.read_csv(MAPPING_PATH, keep_default_na=False)
print(f"loaded {MAPPING_PATH}: {len(mapping)} rows x {mapping.shape[1]} columns")

# --- 1. completeness and field order, against the published data dictionary ----
required = list(zip(dictionary.output_table, dictionary.field_name))
supplied = list(zip(mapping.output_table, mapping.target_field))
assert supplied == required, (
    f"mapping does not match the dictionary. missing={sorted(set(required)-set(supplied))} "
    f"extra={sorted(set(supplied)-set(required))} order_ok={supplied == required}")

# --- 2. stable per-table ids, exactly as the supplied template numbers them -----
# The template pre-fills mapping_id as MAP-<output_table>-<nn>, restarting at 01 for
# each table. Rebuild that sequence rather than trusting the file, so a hand edit that
# renumbers or reorders a row is caught.
expected_ids = [f"MAP-{table}-{n:02d}"
                for table in dictionary.output_table.drop_duplicates()
                for n in range(1, (dictionary.output_table == table).sum() + 1)]
assert list(mapping.mapping_id) == expected_ids, (
    f"mapping_id must follow the template scheme MAP-<output_table>-<nn>; "
    f"first mismatch at {next((i for i, (a, b) in enumerate(zip(mapping.mapping_id, expected_ids)) if a != b), None)}")

# --- 3. no blank cell reaches the submitted file -------------------------------
blank = mapping[(mapping == "").any(axis=1) | mapping.isna().any(axis=1)]
assert blank.empty, f"blank mapping cells:\n{blank}"

# --- 4. source_format uses only the published vocabulary -----------------------
allowed_formats = {"JSON", "XML", "both", "derived"}
bad_format = set(mapping.source_format) - allowed_formats
assert not bad_format, f"source_format values outside the published vocabulary: {bad_format}"

# --- 5. every path token resolves against the structural profile in T1-P2 ------
known_paths = set(structure_profile["structural_path"])
unresolved = [(row.mapping_id, column, token)
              for row in mapping.itertuples()
              for column in ("json_source_path", "xml_source_path")
              for cell in [getattr(row, column)] if cell != "N/A"
              for token in [t.strip() for t in cell.split("|")]
              if token not in known_paths]
assert not unresolved, f"path tokens not found in T1-P2: {unresolved[:5]}"

# --- 6. claims in the text must match what T1-P4 and T1-P6 actually measured ----
# String containment only - no regex, keeping the boundary of T1-P8 intact.
def expected_domain_phrase(table, field):
    values = CATEGORICAL_DOMAINS[(table, field)]
    if len(values) == 1:
        return f"Single observed value {values[0]}."
    return "Observed domain {" + ", ".join(values) + "}."

stale_domain = [(row.mapping_id, f"{row.output_table}.{row.target_field}")
                for row in mapping.itertuples()
                if (row.output_table, row.target_field) in CATEGORICAL_DOMAINS
                and expected_domain_phrase(row.output_table, row.target_field)
                not in row.transformation_or_derivation]
assert not stale_domain, f"domain claims disagree with the T1-P4 measurement: {stale_domain[:5]}"

measured = overlap_profile.set_index("target_table")
def expected_overlap_opening(table, key_name):
    m = measured.loc[table]
    if int(m.json_records) == 0 or int(m.xml_records) == 0:
        return f"Single-source attribute:"
    return (f"Reconcile on {key_name}: {int(m.within_json_repeat_pairs):,} repeated JSON rows and "
            f"{int(m.within_xml_repeat_pairs):,} repeated XML rows inside their own export, plus "
            f"{int(m.keys_in_both_sources):,} keys published by both exports, collapse to "
            f"{int(m.canonical_keys_expected):,} canonical rows.")

stale_overlap = [(row.mapping_id, row.output_table)
                 for row in mapping.itertuples()
                 if expected_overlap_opening(row.output_table, PRIMARY_KEYS[row.output_table])
                 not in row.overlap_or_conflict_rule]
assert not stale_overlap, f"overlap counts disagree with the T1-P6 measurement: {stale_overlap[:5]}"

print(f"  1. {len(mapping)}/{len(dictionary)} required target fields, dictionary order preserved")
print(f"  2. mapping_id {expected_ids[0]}..{expected_ids[-1]} (template scheme)")
print( "  3. no blank cell")
print(f"  4. source_format vocabulary: {sorted(set(mapping.source_format))}")
print(f"  5. all path tokens resolve against the {len(known_paths)} profiled structural paths")
print(f"  6. {len(CATEGORICAL_DOMAINS)} domain claims and {len(measured)} overlap rules match the measurements")
mapping.head(12)

loaded Group005_source_to_target_mapping.csv: 111 rows x 9 columns
  1. 111/111 required target fields, dictionary order preserved
  2. mapping_id MAP-orders-01..MAP-product_reviews-21 (template scheme)
  3. no blank cell
  4. source_format vocabulary: ['JSON', 'XML', 'both', 'derived']
  5. all path tokens resolve against the 201 profiled structural paths
  6. 41 domain claims and 6 overlap rules match the measurements


,mapping_id,output_table,target_field,source_format,json_source_path,xml_source_path,transformation_or_derivation,overlap_or_conflict_rule,notebook_evidence
0,MAP-orders-01,orders,order_id,both,$.orders[*].header.orderID,/OperationsExport/Orders/Order/Header/Order_ID,Trim only. Identifier case and leading zeros preserved (HORD######); used as the order...,Reconcile on order_id: 68 repeated JSON rows and 68 repeated XML rows inside their own...,T1-P1 | T1-P3 | T1-P6
1,MAP-orders-02,orders,source_system_record_id,both,$.orders[*].header.sourceSystemRecordID,/OperationsExport/Orders/Order/Header/Source_System_Record_ID,Trim only; case and hyphenation preserved (SRC-005-H-######). Carried as a source line...,Reconcile on order_id: 68 repeated JSON rows and 68 repeated XML rows inside their own...,T1-P2 | T1-P4 | T1-P6
2,MAP-orders-03,orders,customer_id,both,$.orders[*].header.customerID,/OperationsExport/Orders/Order/Header/Customer_ID,Trim only; leading zeros preserved (CUS#####). Foreign key to customers.customer_id.,Reconcile on order_id: 68 repeated JSON rows and 68 repeated XML rows inside their own...,T1-P3 | T1-P7
3,MAP-orders-04,orders,order_timestamp,both,$.orders[*].header.orderTimestamp,/OperationsExport/Orders/Order/Header/Order_Timestamp,JSON is already YYYY-MM-DD HH:MM:SS; XML is DD/MM/YYYY HH:MM:SS (day component reaches...,Reconcile on order_id: 68 repeated JSON rows and 68 repeated XML rows inside their own...,T1-P2 | T1-P4
4,MAP-orders-05,orders,sales_channel,both,$.orders[*].header.salesChannel,/OperationsExport/Orders/Order/Header/Sales_Channel,"Trim only; structured category retained in published case. Observed domain {Mobile, St...",Reconcile on order_id: 68 repeated JSON rows and 68 repeated XML rows inside their own...,T1-P2 | T1-P4 | T1-P6
5,MAP-orders-06,orders,payment_method,both,$.orders[*].header.paymentMethod,/OperationsExport/Orders/Order/Header/Payment_Method,"Trim only; internal spacing kept. Observed domain {Bank Transfer, Card, Gift Card, Pay...",Reconcile on order_id: 68 repeated JSON rows and 68 repeated XML rows inside their own...,T1-P2 | T1-P4 | T1-P6
6,MAP-orders-07,orders,currency,both,$.orders[*].header.currency,/OperationsExport/Orders/Order/Header/Currency,Trim and upper-case ISO code; single observed value AUD. Cross-checked against the AUD...,Reconcile on order_id: 68 repeated JSON rows and 68 repeated XML rows inside their own...,T1-P2 | T1-P4
7,MAP-orders-08,orders,nearest_warehouse,both,$.orders[*].header.nearestWarehouse,/OperationsExport/Orders/Order/Header/Nearest_Warehouse,"Trim only; warehouse name kept in published case. Observed domain {Bakers, Nickolson, ...",Reconcile on order_id: 68 repeated JSON rows and 68 repeated XML rows inside their own...,T1-P2 | T1-P4 | T1-P6
8,MAP-orders-09,orders,order_status,both,$.orders[*].header.orderStatus,/OperationsExport/Orders/Order/Header/Order_Status,Trim only. Single observed value Completed.,Reconcile on order_id: 68 repeated JSON rows and 68 repeated XML rows inside their own...,T1-P2 | T1-P4 | T1-P6
9,MAP-orders-10,orders,order_price,derived,$.orders[*].shoppingCart[*].quantity | $.orders[*].shoppingCart[*].unitPrice | $.order...,/OperationsExport/Orders/Order/Shopping_Cart/Item/Quantity | /OperationsExport/Orders/...,Derived as step 2 of the published order arithmetic: the sum over the order's cart lin...,Reconcile on order_id: 68 repeated JSON rows and 68 repeated XML rows inside their own...,T1-P4 | T1-P6 | T1-P9


In [26]:
summary = (mapping.groupby(["output_table", "source_format"]).size()
           .unstack(fill_value=0)
           .reindex(columns=["JSON", "XML", "both", "derived"], fill_value=0))
summary["total"] = summary.sum(axis=1)
summary = summary.reindex(["orders", "order_items", "customers", "deliveries", "products", "product_reviews"])
summary.loc["TOTAL"] = summary.sum()
save_profile(summary.reset_index(), "mapping_source_format_summary")
summary

[saved] profiling/Group005_T1_mapping_source_format_summary.csv  (7 rows)


source_format,JSON,XML,both,derived,total
output_table,,,,,
orders,0,0,18,5,23
order_items,0,0,5,1,6
customers,20,0,0,0,20
deliveries,0,0,20,0,20
products,0,20,0,1,21
product_reviews,0,0,14,7,21
TOTAL,20,20,57,14,111


## 3. Text and regex functions

### 3.1 Cleaning and extraction implementation

---
#### `T1-P8` — Narrative field profile (inputs to the Task 3 regex contract)

The bounded narrative fields are profiled for the markers, markup, URLs, entities, emoji and
scripts they actually contain. Regular expressions appear here for the first time, applied to
values already extracted by the structured parsers — and written in the verbose `(?x)` style
of the Week 4 applied session. The cleaning functions themselves belong to Task 3.

In [27]:
MARKERS = ["[SYSTEM]", "[CATALOGUE]", "[VERIFIED_PURCHASE]", "[SOURCE:", "[RATING:",
           "#verified-buyer", "@store_support", "PROMO:", "Reference:", "SKU:"]

narrative_pools = {
    "orders.customer_note":           columns_of((json_headers, "customerNote"), (xml_headers, "Customer_Note")),
    "product_reviews.review_text":    columns_of((json_reviews, "reviewText"), (xml_reviews, "Review_Text")),
    "products.product_description":   columns_of((xml_products, "Product_Description")),
    "deliveries.delivery_note_clean": columns_of((json_deliveries, "deliveryNoteClean"), (xml_deliveries, "Delivery_Note_Clean")),
    "product_reviews.review_title":   columns_of((json_reviews, "reviewTitle"), (xml_reviews, "Review_Title")),
}

TAG_RE = re.compile(r"</?([a-zA-Z][a-zA-Z0-9]*)")
ENTITY_RE = re.compile(r"&[a-zA-Z]{2,8};|&#\d{2,5};")

def has_non_latin_letter(text):
    for char in text:
        if unicodedata.category(char).startswith("L") and "LATIN" not in unicodedata.name(char, ""):
            return True
    return False

narrative_rows = []
for field, pool in narrative_pools.items():
    pool = pool.dropna().astype(str)
    joined = " ".join(pool)
    narrative_rows.append({
        "narrative_field": field,
        "values_profiled": len(pool),
        "markers_present": ", ".join(m for m in MARKERS if m in joined) or "none",
        "html_tags": ", ".join(sorted(set(TAG_RE.findall(joined)))) or "none",
        "html_entities_after_parse": ", ".join(sorted(set(ENTITY_RE.findall(joined)))) or "none",
        "with_url": int(pool.str.contains("http", regex=False).sum()),
        "with_emoji_So": int(pool.map(lambda t: any(unicodedata.category(c) == "So" for c in t)).sum()),
        "with_non_ascii": int(pool.map(lambda t: any(ord(c) > 127 for c in t)).sum()),
        "with_non_latin_letter": int(pool.map(has_non_latin_letter).sum()),
    })
narrative_profile = pd.DataFrame(narrative_rows)
save_profile(narrative_profile, "narrative_field_profile")
narrative_profile

[saved] profiling/Group005_T1_narrative_field_profile.csv  (5 rows)


,narrative_field,values_profiled,markers_present,html_tags,html_entities_after_parse,with_url,with_emoji_So,with_non_ascii,with_non_latin_letter
0,orders.customer_note,5636,"[SYSTEM], PROMO:",p,none,5636,0,0,0
1,product_reviews.review_text,7892,"[VERIFIED_PURCHASE], [SOURCE:, [RATING:, #verified-buyer, @store_support, Reference:, ...","article, aside, blockquote, br, div, p, section",&nbsp;,7892,7892,7892,315
2,products.product_description,1000,[CATALOGUE],section,none,1000,0,0,0
3,deliveries.delivery_note_clean,5636,none,none,none,0,0,0,0
4,product_reviews.review_title,7892,none,none,none,0,0,541,315


In [28]:
# Embedded business references, counted on the raw narrative with pandas .str + verbose regex.
ORDER_REF_RE = r'''(?x)
    \b            # word boundary, so a longer near-match is rejected
    (?P<prefix>[HC])ORD
    \d{6}         # exactly six digits
    \b
'''
SKU_RE = r'''(?x)
    \b SKU-
    (?P<token>[A-Za-z0-9]+)
    \b
'''
PROMO_RE = r'''(?x)
    \b (?P<band>B[1-5]) SAVE-
    \d{2}         # exactly two digits
    \b
'''

review_pool = narrative_pools["product_reviews.review_text"].dropna().astype(str)
note_pool = narrative_pools["orders.customer_note"].dropna().astype(str)

reference_profile = pd.DataFrame([
    {"reference": "order reference", "published_format": "HORD or CORD + exactly 6 digits",
     "source_field": "product_reviews.review_text",
     "values_carrying_it": int(review_pool.str.extract(ORDER_REF_RE)["prefix"].notna().sum()),
     "prefixes_observed": {key: int(value) for key, value in
                           review_pool.str.extract(ORDER_REF_RE)["prefix"].value_counts().items()},
     "target_field": "extracted_order_reference"},
    {"reference": "product SKU", "published_format": "SKU- + one or more ASCII letters/digits",
     "source_field": "product_reviews.review_text",
     "values_carrying_it": int(review_pool.str.extract(SKU_RE)["token"].notna().sum()),
     "prefixes_observed": {key: int(value) for key, value in
                           review_pool.str.extract(SKU_RE)["token"].str[:3].value_counts().items()},
     "target_field": "extracted_product_sku"},
    {"reference": "promotion code", "published_format": "B1SAVE- to B5SAVE- + exactly 2 digits",
     "source_field": "orders.customer_note",
     "values_carrying_it": int(note_pool.str.extract(PROMO_RE)["band"].notna().sum()),
     "prefixes_observed": {key: int(value) for key, value in
                           note_pool.str.extract(PROMO_RE)["band"].value_counts().items()},
     "target_field": "orders.promo_code"},
])
save_profile(reference_profile, "embedded_reference_profile")
reference_profile

[saved] profiling/Group005_T1_embedded_reference_profile.csv  (3 rows)


,reference,published_format,source_field,values_carrying_it,prefixes_observed,target_field
0,order reference,HORD or CORD + exactly 6 digits,product_reviews.review_text,7892,{'H': 7892},extracted_order_reference
1,product SKU,SKU- + one or more ASCII letters/digits,product_reviews.review_text,7892,"{'VEL': 3965, 'CAN': 3927}",extracted_product_sku
2,promotion code,B1SAVE- to B5SAVE- + exactly 2 digits,orders.customer_note,2103,{'B1': 2103},orders.promo_code


**Material narrative findings**

* `orders.customer_note` carries `[SYSTEM]`, `<p>` markup, an optional `PROMO: <code>` wrapper
  and a help URL — no emoji, no non-ASCII text.
* `products.product_description` carries `[CATALOGUE]`, `<section>` markup and a catalogue URL.
* `product_reviews.review_text` is the noisy field: six HTML-like tags, all seven
  bracketed/social markers, a URL, emoji in **every** value, and a complete
  `Reference: <order> SKU: <sku>` wrapper on every value. `&nbsp;` survives XML parsing and
  must be decoded with `html.unescape`, not stripped as literal text.
* Non-Latin scripts (Chinese, Devanagari, Arabic, Japanese, Cyrillic and others) appear in the
  review bodies. `review_body_clean` must preserve them; `review_body_latin_analysis` is a
  **separate** derived field. A non-ASCII character is not automatically non-Latin.
* `deliveries.delivery_note_clean` and `product_reviews.review_title` arrive already clean.
* Only the `HORD` order-reference prefix occurs in this package, but the `CORD` alternative is
  published in the contract and is therefore accepted by the extractor.

In [29]:
from Group005_text_functions import (
    build_latin_analysis,
    clean_narrative_text,
    contains_non_latin_script,
    extract_order_reference,
    extract_product_sku,
    extract_promo_code,
)

---
#### Check: regex stays inside its assessed boundary

The specification allows regex only for bounded narrative work, never for document structure.
This check re-reads the notebook's own source and reports which sections contain a regular
expression, so the claim is verifiable rather than asserted.

In [30]:
import io, json as _json

NOTEBOOK_PATH = f"{GROUP_ID}_solution.ipynb"
REGEX_MARKERS = ("re.compile", "re.sub(", "re.findall(", "re.match(", "re.search(", "regex=True")

if os.path.exists(NOTEBOOK_PATH):
    with io.open(NOTEBOOK_PATH, encoding="utf-8") as fh:
        cells = _json.load(fh)["cells"]
    section, rows = "(preamble)", []
    for cell in cells:
        source = "".join(cell["source"])
        if cell["cell_type"] == "markdown":
            for line in source.splitlines():
                if line.lstrip("# ").startswith("`T1-"):
                    section = line.strip("# ").split("`")[1]
        else:
            if "REGEX_MARKERS" in source:
                continue          # the checking cell names the markers, so skip itself
            hits = [m for m in REGEX_MARKERS if m in source]
            if hits:
                rows.append({"section": section, "regex_constructs": ", ".join(sorted(set(hits)))})
    regex_usage = pd.DataFrame(rows).drop_duplicates()
    print("sections containing a regular expression:", sorted(regex_usage["section"].unique()))
    print("structural parsing sections (T1-P1, T1-P2) clean:",
          not regex_usage["section"].isin(["T1-P1", "T1-P2"]).any())
    print(regex_usage)
else:
    print("notebook file not on disk (running as the exported script) - check skipped")

sections containing a regular expression: ['T1-P8']
structural parsing sections (T1-P1, T1-P2) clean: True
  section regex_constructs
0   T1-P8       re.compile


### 3.2 Public and student-designed tests

Three design decisions in `Group005_text_functions.py` decide most of the boundary cases, so
they are stated here before the evidence.

**Markers are removed from an explicit list, never by a general bracket pattern.** The
specification publishes exactly seven removable markers. A pattern such as `\[.*?\]` would also
delete `[NOTE]` or `[URGENT]`, which is customer wording and must survive. Case `TXT-18` tests
precisely this.

**Reference extraction rejects near-matches instead of truncating them.** A word boundary alone
is not enough for a SKU: in `SKU-ABC123-extra` the hyphen counts as a boundary, so `\b` would
return the valid-looking prefix `SKU-ABC123`. The lookarounds exclude `-`, so the malformed
value is rejected outright, as `TXT-12` requires.

**Emoji are identified by Unicode block, not by category `So`.** Category `So` is much broader
than emoji: it also holds `©`, `®`, `™` and `°`, which are ordinary text symbols a customer may
have typed and which the specification never asks to remove. The published narratives use
`©` and `™` only inside `[SOURCE: ...]` markers, so both definitions agree on this package —
but they disagree on any private case that puts one in the body text.

#### Public cases

The unit supplies `templates/A1_public_text_test_cases.csv`. It is loaded with
`keep_default_na=False`; without that, pandas reads the expected value `NaN` as a missing cell
and every sentinel case compares incorrectly.

Every case is shown in full — input, expected, observed and status — so the evidence can be read
without rerunning anything.

In [31]:
PUBLIC_CASES_PATH = os.path.join(TEMPLATE_DIR, "A1_public_text_test_cases.csv")
public_cases = pd.read_csv(PUBLIC_CASES_PATH, keep_default_na=False)

TEXT_FUNCTIONS = {
    "clean_narrative_text": clean_narrative_text,
    "extract_order_reference": extract_order_reference,
    "extract_product_sku": extract_product_sku,
    "extract_promo_code": extract_promo_code,
    "build_latin_analysis": build_latin_analysis,
    "contains_non_latin_script": contains_non_latin_script,
}

public_results = pd.DataFrame([
    {
        "case_id": row.case_id,
        "function": row.function,
        "input_value": row.input_value,
        "expected": row.expected_output,
        "observed": str(TEXT_FUNCTIONS[row.function](row.input_value)),
        "purpose": row.purpose,
    }
    for row in public_cases.itertuples()
]).assign(status=lambda t: (t.expected == t.observed).map({True: "PASS", False: "FAIL"}))

print(f"VAL-TEXT-01  public test cases: "
      f"{int((public_results.status == 'PASS').sum())}/{len(public_results)} PASS")
print(public_results.groupby("function").status.value_counts().unstack(fill_value=0).to_string())

with pd.option_context("display.max_colwidth", 76):
    display(public_results[["case_id", "function", "input_value", "expected", "observed", "status"]])

VAL-TEXT-01  public test cases: 18/18 PASS
status                     PASS
function                       
build_latin_analysis          2
clean_narrative_text          8
contains_non_latin_script     1
extract_order_reference       3
extract_product_sku           2
extract_promo_code            2


,case_id,function,input_value,expected,observed,status
0,TXT-01,clean_narrative_text,[SYSTEM] <p>Leave at reception</p> PROMO: B3SAVE-24 https://orders.examp...,leave at reception,leave at reception,PASS
1,TXT-02,extract_promo_code,[SYSTEM] <p>Leave at reception</p> PROMO: B3SAVE-24,B3SAVE-24,B3SAVE-24,PASS
2,TXT-03,clean_narrative_text,[SOURCE: mobile-app] <p>Caf&eacute; setup was easy 😊</p> Reference: HORD...,café setup was easy,café setup was easy,PASS
3,TXT-04,extract_order_reference,Reference: HORD123456 | SKU: SKU-ABC123,HORD123456,HORD123456,PASS
4,TXT-05,extract_product_sku,Reference: HORD123456 | SKU: SKU-ABC123,SKU-ABC123,SKU-ABC123,PASS
5,TXT-06,extract_order_reference,codes XHORD1234567 and HORD12345 are invalid,NaN,NaN,PASS
6,TXT-07,build_latin_analysis,service était bon 包装很好,service était bon,service était bon,PASS
7,TXT-08,contains_non_latin_script,service était bon 包装很好,True,True,PASS
8,TXT-09,build_latin_analysis,包装很好,NaN,NaN,PASS
9,TXT-10,clean_narrative_text,[VERIFIED_PURCHASE] <div>Reliable for daily use</div>,reliable for daily use,reliable for daily use,PASS


**Observed result: 18/18 PASS. Status: PASS.** Three cases decide more than they look.

`TXT-18` supplies `[NOTE] Keep bracketed customer wording` and expects
`[note] keep bracketed customer wording`. The brackets survive because `[NOTE]` is not one of the
seven published markers; only the case changes.

`TXT-12` supplies `SKU-ABC123-extra` and expects the sentinel rather than `SKU-ABC123`. Returning
the prefix would be a silent data error, not partial credit.

`TXT-03` combines a parameterised `[SOURCE: ...]` marker, an HTML entity, an emoji, the full
`Reference: … | SKU: …` wrapper and a social token, and expects `café setup was easy`. The `é`
proves entity decoding runs before tag stripping and that accented Latin text is never treated
as foreign.

#### Student-designed cases

The public set leaves gaps the specification explicitly asks about: *"Document and test your text
functions with matched, unmatched, missing, multilingual and near-match examples."* It contains
no `None` input, no case for the `NaN` sentinel behaviour recorded in `ASM-13`, and only one case
for `contains_non_latin_script`.

The 61 cases below close those gaps, grouped by the five categories the specification names plus
the sentinel and symbol behaviour this package relies on.

In [32]:
OWN_CASES = [
    ("missing",      "None input to the cleaner",                clean_narrative_text, (None,), MISSING),
    ("missing",      "None input to the order extractor",        extract_order_reference, (None,), MISSING),
    ("missing",      "None input to the SKU extractor",          extract_product_sku, (None,), MISSING),
    ("missing",      "None input to the promo extractor",        extract_promo_code, (None,), MISSING),
    ("missing",      "whitespace-only input",                    clean_narrative_text, ("   ",), MISSING),
    ("missing",      "marker and emoji leave nothing readable",  clean_narrative_text, ("[SYSTEM] \U0001F60A",), MISSING),

    ("near-match",   "order reference with seven digits",        extract_order_reference, ("HORD1234567",), MISSING),
    ("near-match",   "order reference with five digits",         extract_order_reference, ("HORD12345",), MISSING),
    ("near-match",   "order reference with letters",             extract_order_reference, ("HORDABCDEF",), MISSING),
    ("near-match",   "order reference embedded in a word",       extract_order_reference, ("XHORD123456",), MISSING),
    ("near-match",   "promotion band outside B1-B5",             extract_promo_code, ("B6SAVE-12",), MISSING),
    ("near-match",   "promotion code with three digits",         extract_promo_code, ("B1SAVE-123",), MISSING),
    ("near-match",   "promotion code with one digit",            extract_promo_code, ("B1SAVE-1",), MISSING),
    ("near-match",   "SKU embedded in a longer token",           extract_product_sku, ("XSKU-ABC1",), MISSING),
    ("near-match",   "SKU prefix with no body",                  extract_product_sku, ("SKU-",), MISSING),

    ("matched",      "CORD prefix accepted, not only HORD",      extract_order_reference, ("ref CORD000001 here",), "CORD000001"),
    ("matched",      "lower-case reference normalised to upper", extract_order_reference, ("reference: hord123456",), "HORD123456"),
    ("matched",      "lower-case SKU normalised to upper",       extract_product_sku, ("SKU: sku-can00871",), "SKU-CAN00871"),
    ("matched",      "promotion code inside its wrapper",        extract_promo_code, ("PROMO: B5SAVE-09",), "B5SAVE-09"),

    ("unmatched",    "no reference present at all",              extract_order_reference, ("no reference at all",), MISSING),
    ("unmatched",    "note carrying no promotion code",          extract_promo_code, ("[SYSTEM] no special instruction",), MISSING),

    ("multilingual", "accented Latin is not non-Latin",          contains_non_latin_script, ("café niño über",), False),
    ("multilingual", "plain ASCII is not non-Latin",             contains_non_latin_script, ("hello world",), False),
    ("multilingual", "Cyrillic detected",                        contains_non_latin_script, ("привет",), True),
    ("multilingual", "Arabic detected",                          contains_non_latin_script, ("مرحبا",), True),
    ("multilingual", "diacritics survive the Latin analysis",    build_latin_analysis, ("café niño über",), "café niño über"),
    ("multilingual", "CJK letters dropped, Latin kept",          build_latin_analysis, ("good 很好 product",), "good product"),
    ("multilingual", "CJK punctuation dropped with its script",  build_latin_analysis, ("service était bon 包装很好。",), "service était bon"),
    ("multilingual", "Latin typography survives",                build_latin_analysis, ("«cite» — dash 很好",), "«cite» — dash"),
    ("multilingual", "cleaner preserves multilingual text",      clean_narrative_text, ("<p>我用candle quest 768</p>",), "我用candle quest 768"),

    ("sentinel",     "sentinel contains only Latin letters",     contains_non_latin_script, (MISSING,), False),
    ("symbol",       "trade mark sign is not an emoji",          clean_narrative_text, ("<p>Product™ good</p>",), "product™ good"),
    ("symbol",       "degree sign is not an emoji",              clean_narrative_text, ("<p>at 25° today</p>",), "at 25° today"),
    ("boundary",     "unpublished bracketed wording preserved",  clean_narrative_text, ("[URGENT] please deliver",), "[urgent] please deliver"),
    ("multilingual", "Vietnamese diacritics are Latin, not foreign", contains_non_latin_script, ("s\u1ea3n ph\u1ea9m r\u1ea5t t\u1ed1t",), False),
    ("multilingual", "Vietnamese survives the Latin analysis", build_latin_analysis, ("caf\u00e9 r\u1ea5t t\u1ed1t",), "caf\u00e9 r\u1ea5t t\u1ed1t"),
    ("multilingual", "Japanese alone leaves no Latin letter", build_latin_analysis, ("\u826f\u3044\u5546\u54c1",), MISSING),

    ("symbol",       "emoji variation sequence removed whole", clean_narrative_text, ("great \u00a9\ufe0f product",), "great product"),
    ("symbol",       "wavy dash with variation selector removed", clean_narrative_text, ("great \u3030\ufe0f product",), "great product"),
    ("symbol",       "bare wavy dash is text, not emoji", clean_narrative_text, ("great \u3030 dash",), "great \u3030 dash"),
    ("symbol",       "keycap emoji removed with its base digit", clean_narrative_text, ("rating 1\ufe0f\u20e3 good",), "rating good"),

    ("boundary",     "emoji inside the promo wrapper still matches", clean_narrative_text, ("PROMO: \U0001F60A B1SAVE-12",), MISSING),
    ("boundary",     "emoji inside the reference wrapper still matches", clean_narrative_text, ("Reference: \U0001F60A HORD123456 | SKU: SKU-ABC123",), MISSING),
    ("near-match",   "decomposed diacritic before a reference", extract_order_reference, ("e\u0301HORD123456",), MISSING),
    ("near-match",   "decomposed diacritic after a reference", extract_order_reference, ("HORD123456\u0301",), MISSING),
    ("near-match",   "CJK letter before a SKU", extract_product_sku, ("\u5305SKU-ABC123",), MISSING),
    ("near-match",   "Arabic-Indic digits are not ASCII digits", extract_order_reference, ("HORD\u0661\u0662\u0663\u0664\u0665\u0666",), MISSING),
    ("multilingual", "readable wording between the halves is not a separator", clean_narrative_text, ("Reference: HORD123456 \u6ce8\u610f SKU: SKU-ABC123",), "reference: hord123456 \u6ce8\u610f sku: sku-abc123"),
    ("boundary",     "an incomplete wrapper is left as customer text", clean_narrative_text, ("Reference: HORD123456",), "reference: hord123456"),
    ("boundary",     "a malformed SKU half leaves the whole wrapper intact", clean_narrative_text, ("Reference: HORD123456 | SKU: SKU-ABC123-extra",), "reference: hord123456 | sku: sku-abc123-extra"),
    ("boundary",     "an em dash is a valid wrapper separator", clean_narrative_text, ("Reference: HORD123456 \u2014 SKU: SKU-ABC123",), MISSING),
    ("near-match",   "CJK after a promo code blocks the wrapper", clean_narrative_text, ("PROMO: B1SAVE-12\u4e2d",), "promo: b1save-12\u4e2d"),
    ("near-match",   "combining mark after a promo code blocks it", clean_narrative_text, ("PROMO: B1SAVE-12\u0301",), "promo: b1save-12\u0301"),
    ("near-match",   "long s must not case-fold into the SKU prefix", extract_product_sku, ("\u017fKU-ABC123",), MISSING),
    ("near-match",   "long s must not case-fold inside B1SAVE", extract_promo_code, ("B1\u017fAVE-12",), MISSING),
    ("near-match",   "long s is not an ASCII SKU character", extract_product_sku, ("SKU-\u017f",), MISSING),
    ("near-match",   "dotless i is not an ASCII SKU character", extract_product_sku, ("SKU-\u0131",), MISSING),
    ("near-match",   "dotted capital I is not an ASCII SKU character", extract_product_sku, ("SKU-\u0130",), MISSING),
    ("near-match",   "the Kelvin sign is not an ASCII SKU character", extract_product_sku, ("SKU-\u212a",), MISSING),
    ("boundary",     "a marker spelled with a non-ASCII letter is not published", clean_narrative_text, ("[VER\u0130FIED_PURCHASE] Keep",), "[ver\u0069\u0307fied_purchase] keep"),
    ("boundary",     "a social token spelled with dotless i is not published", clean_narrative_text, ("#ver\u0131fied-buyer Keep",), "#ver\u0131fied-buyer keep"),
]

own_results = pd.DataFrame([
    {"case_id": f"OWN-{i:02d}", "category": category, "description": description,
     "input_value": repr(args[0]), "expected": repr(expected), "observed": repr(function(*args))}
    for i, (category, description, function, args, expected) in enumerate(OWN_CASES, start=1)
]).assign(status=lambda t: (t.expected == t.observed).map({True: "PASS", False: "FAIL"}))

print(f"VAL-TEXT-02  student-designed cases: "
      f"{int((own_results.status == 'PASS').sum())}/{len(own_results)} PASS")
print(own_results.groupby("category").status.value_counts().unstack(fill_value=0).to_string())

with pd.option_context("display.max_colwidth", 54):
    display(own_results)

VAL-TEXT-02  student-designed cases: 61/61 PASS
status        PASS
category          
boundary         8
matched          4
missing          6
multilingual    13
near-match      21
sentinel         1
symbol           6
unmatched        2


,case_id,category,description,input_value,expected,observed,status
0,OWN-01,missing,None input to the cleaner,None,'NaN','NaN',PASS
1,OWN-02,missing,None input to the order extractor,None,'NaN','NaN',PASS
2,OWN-03,missing,None input to the SKU extractor,None,'NaN','NaN',PASS
3,OWN-04,missing,None input to the promo extractor,None,'NaN','NaN',PASS
4,OWN-05,missing,whitespace-only input,' ','NaN','NaN',PASS
...,...,...,...,...,...,...,...
56,OWN-57,near-match,dotless i is not an ASCII SKU character,'SKU-ı','NaN','NaN',PASS
57,OWN-58,near-match,dotted capital I is not an ASCII SKU character,'SKU-İ','NaN','NaN',PASS
58,OWN-59,near-match,the Kelvin sign is not an ASCII SKU character,'SKU-K','NaN','NaN',PASS
59,OWN-60,boundary,a marker spelled with a non-ASCII letter is not pu...,'[VERİFIED_PURCHASE] Keep','[veri̇fied_purchase] keep','[veri̇fied_purchase] keep',PASS


**Observed result: 61/61 PASS. Status: PASS.**

The `near-match` group is deliberately the largest. Rejection is where a plausible-looking
implementation quietly fails: every one of those nine inputs *looks* like a valid reference and
every one must return the sentinel instead.

The `multilingual` group encodes the distinction the specification is most explicit about —
*"A non-ASCII character is not automatically non-Latin"*. `café niño über` is non-ASCII in places
yet entirely Latin, so `contains_non_latin_script` returns `False` and `build_latin_analysis`
returns it unchanged. Cyrillic and Arabic return `True`. The cleaner keeps
`我用candle quest 768` intact, because `review_body_clean` must preserve valid multilingual UTF-8
rather than erase it.

The `symbol` group guards the emoji definition. `Product™` and `25°` survive cleaning because
they are text symbols, not emoji — a distinction that Unicode category `So` alone does not make.

#### Derived review measures

The specification defines two measures the six published functions do not return, and fixes
their behaviour when the cleaned body is the sentinel:

> `review_length_chars` = number of Python characters in `review_body_clean`
> `review_word_count` = number of whitespace-separated tokens in `review_body_clean`
> *"When `review_body_clean` is the literal `NaN`, preserve the published sentinel behaviour
> rather than counting the three letters as an ordinary review."*

`ASM-13` records the reading applied here: the measures are `0` and `0`, not `3` and `1`, and
`contains_non_latin_script` is `False`. They are always emitted as a number and a boolean, never
as the string `NaN`, because the dictionary types those three fields as numeric, numeric and
boolean with `nullable=False`.

In [33]:
def review_length_chars(cleaned_body):
    '''Characters in review_body_clean; 0 when the body is the published sentinel.'''
    return 0 if cleaned_body == MISSING else len(cleaned_body)

def review_word_count(cleaned_body):
    '''Whitespace-separated tokens in review_body_clean; 0 for the sentinel.'''
    return 0 if cleaned_body == MISSING else len(cleaned_body.split())

measure_demo = pd.DataFrame([
    {"review_body_clean": body,
     "review_length_chars": review_length_chars(body),
     "review_word_count": review_word_count(body),
     "contains_non_latin_script": contains_non_latin_script(body),
     "note": note}
    for body, note in [
        ("the candle shift 970 is reliable", "ordinary English review"),
        ("我用candle quest 768", "multilingual review, letters preserved"),
        ("café était bon", "accented Latin only"),
        (MISSING, "sentinel: counted as 0/0/False, not 3/1"),
    ]
])
print("VAL-TEXT-03  sentinel behaviour of the derived review measures")
display(measure_demo)

VAL-TEXT-03  sentinel behaviour of the derived review measures


,review_body_clean,review_length_chars,review_word_count,contains_non_latin_script,note
0,the candle shift 970 is reliable,32,6,False,ordinary English review
1,我用candle quest 768,18,3,True,"multilingual review, letters preserved"
2,café était bon,14,3,False,accented Latin only
3,NaN,0,0,False,"sentinel: counted as 0/0/False, not 3/1"


**Observed result: the sentinel row reports `0`, `0` and `False`. Status: PASS.** The three
letters of `NaN` are not counted as a one-word review, and no numeric or boolean field ever
receives the string.

#### Cross-check against the Task 1 measurements

Passing a supplied test file proves the functions satisfy 18 published examples. It does not
prove they behave correctly on this package's 7,892 review bodies and 5,636 customer notes.

The profiling in `T1-P8` counted those narratives independently, before these functions were
written and through a different code path. Running the functions over the same raw values should
reproduce those counts exactly.

In [34]:
raw_reviews = columns_of((json_reviews, "reviewText"), (xml_reviews, "Review_Text")).dropna().astype(str)
raw_notes = columns_of((json_headers, "customerNote"), (xml_headers, "Customer_Note")).dropna().astype(str)

narrative_measured = pd.read_csv(os.path.join(PROFILE_DIR, f"{GROUP_ID}_T1_narrative_field_profile.csv"))
reference_measured = pd.read_csv(os.path.join(PROFILE_DIR, f"{GROUP_ID}_T1_embedded_reference_profile.csv"))
measured = lambda name: int(reference_measured.loc[reference_measured.reference == name,
                                                   "values_carrying_it"].iloc[0])

cross_check = pd.DataFrame([
    {"quantity": "review bodies containing a non-Latin letter",
     "task_3_functions": int(raw_reviews.map(clean_narrative_text).map(contains_non_latin_script).sum()),
     "measured_in_T1_P8": int(narrative_measured.loc[
         narrative_measured.narrative_field == "product_reviews.review_text",
         "with_non_latin_letter"].iloc[0])},
    {"quantity": "review bodies yielding an order reference",
     "task_3_functions": int((raw_reviews.map(extract_order_reference) != MISSING).sum()),
     "measured_in_T1_P8": measured("order reference")},
    {"quantity": "review bodies yielding a product SKU",
     "task_3_functions": int((raw_reviews.map(extract_product_sku) != MISSING).sum()),
     "measured_in_T1_P8": measured("product SKU")},
    {"quantity": "customer notes yielding a promotion code",
     "task_3_functions": int((raw_notes.map(extract_promo_code) != MISSING).sum()),
     "measured_in_T1_P8": measured("promotion code")},
]).assign(status=lambda t: (t.task_3_functions == t.measured_in_T1_P8).map({True: "PASS", False: "FAIL"}))

# The extracted SKU must also name the product the review is actually about.
review_products = columns_of((json_reviews, "productID"), (xml_reviews, "Product_ID")).astype(str)
catalogue_sku = dict(zip(xml_products["Product_ID"], xml_products["Product_Sku"].str.upper()))
sku_matches = int(sum(catalogue_sku.get(product) == sku for product, sku
                      in zip(review_products, raw_reviews.map(extract_product_sku))))

print("VAL-TEXT-04  extraction reproduces the T1-P8 counts")
print(f"VAL-TEXT-05  extracted SKU names the reviewed product: {sku_matches}/{len(raw_reviews)}")
cross_check

VAL-TEXT-04  extraction reproduces the T1-P8 counts
VAL-TEXT-05  extracted SKU names the reviewed product: 7892/7892


,quantity,task_3_functions,measured_in_T1_P8,status
0,review bodies containing a non-Latin letter,315,315,PASS
1,review bodies yielding an order reference,7892,7892,PASS
2,review bodies yielding a product SKU,7892,7892,PASS
3,customer notes yielding a promotion code,2103,2103,PASS


**Observed result: all four counts agree, and 7,892 of 7,892 extracted SKUs name the reviewed
product. Status: PASS.**

The last line is the strongest of the five. `extract_product_sku` reads only the review text and
has no access to the product catalogue, so a full match against `products.product_sku` for the
referenced `product_id` means the extraction is not merely well-formed but correct.

**Interpretation and limitation.** No narrative in this package cleans to an empty string, so the
sentinel path in `clean_narrative_text` is exercised only by the cases above, never by the data
itself. `ASM-13` records the same gap for the derived review measures. The rule is implemented
and tested against the published contract; it is simply not observed here, and a private test is
the only thing that will exercise it.

## 4. Build the six standardised relational tables


#### Task 2 — Six standardised relational tables

Task 2 turns the profiled source frames into exactly the six tables in the public data
dictionary. Reconciliation is value based: comparable fields are normalised first, every
non-missing value for a business key must agree, and only then are repeated/source-overlap
rows collapsed. No JSON-over-XML or XML-over-JSON precedence rule is used.


##### Data flow and build order

Task 2 runs in four stages, and the order is driven by data dependencies rather than by the
section numbering:

1. **Reconcile** (`T2-R1`) — the four entities that appear in *both* exports are collapsed to
   one canonical row per published key, producing `canonical_orders_source`,
   `canonical_items_source`, `canonical_deliveries_source` and `canonical_reviews_source`.
   Narrative-derived fields are reconciled separately because they must be extracted from the
   raw parser output before cleaning.
2. **Build the transactional grain** (`T2-B2`) — `order_items` is materialised first, because
   `orders.order_price` is an aggregate of the rounded item-level `line_revenue`.
3. **Build the remaining tables** (`T2-B1`, `T2-B3`) — `customers` and `products` are
   single-source; `deliveries` and `product_reviews` come from the canonical frames.
4. **Enforce the schema and export** (`T2-O1`) — every table is reindexed to the public
   dictionary field order, sorted by primary key, and written to CSV.

| Target table | Grain | Built from |
|---|---|---|
| `orders` | one row per canonical order | `canonical_orders_source` + `order_price_by_id` |
| `order_items` | one row per order item | `canonical_items_source` |
| `customers` | one row per customer | `json_customers` (JSON only) |
| `deliveries` | one row per completed order delivery | `canonical_deliveries_source` |
| `products` | one row per product | `xml_products` (XML only) |
| `product_reviews` | one row per canonical product review | `canonical_reviews_source` |

The diagnostics in this section are **lightweight inspection checks** intended to make the
transformation behaviour visible while building and debugging. The formal, citable
`VAL-*` register with PASS/FAIL status and resolutions is Task 4, in section 6.


#### `T2-R1` — Reconcile canonical rows without source precedence


##### Why canonical reconciliation is needed

`T1-P6` established two facts that dictate this step. First, each export repeats a subset of
its own records verbatim. Second, the two exports independently publish an overlapping subset
of orders, items, deliveries and reviews. A target table must therefore contain exactly one
row per published business key, drawn from both sources without preferring either.

The comparable fields were already normalised in `T1-P6` (`NORMALISED`), so `AUD 5,287.78`
and `5287.78`, or `Y` and `true`, are already the same Python value before any comparison
happens here. This matters: reconciling on raw values would report spurious conflicts caused
purely by notation.

`task2_conflict_rows` accumulates every unresolved disagreement encountered anywhere in this
section, and is turned into the `task2_conflicts` evidence table at the end.


In [ ]:
task2_conflict_rows = []

def _non_missing_values(series):
    '''Return distinct non-missing values while preserving their parsed Python types.'''
    values = []
    for value in series:
        if is_absent(value):
            continue
        if not any(value == observed for observed in values):
            values.append(value)
    return values


##### Canonical record reconciliation strategy

`reconcile_canonical()` groups the concatenated normalised frames by the published primary key
and resolves each field independently, applying three rules:

* **one populated value fills a missing counterpart** — if one export omits a field the other
  supplies, the populated value is retained;
* **equivalent populated values collapse to one canonical value** — after normalisation the
  repeated records agree, so a single value survives;
* **conflicting non-missing values are recorded, not resolved** — the disagreement is written
  to `task2_conflict_rows`, the field is left absent, and Task 4's `VAL-CONFLICT-01` carries
  the failure forward.

The third rule is the important design decision. Taking the first value would silently turn
the concatenation order into a JSON-over-XML precedence rule that no evidence supports, so the
value is deliberately left unresolved and made visible instead.


In [ ]:
def reconcile_canonical(table):
    '''Coalesce normalised source frames to one canonical row per published key.

    One populated value may fill a missing counterpart. Two different populated values
    are recorded and rejected rather than silently resolved by source priority.
    '''
    key = PRIMARY_KEYS[table]
    frames = [NORMALISED[(table, source)] for source in ("JSON", "XML")
              if (table, source) in NORMALISED and not NORMALISED[(table, source)].empty]
    combined = pd.concat(frames, ignore_index=True)
    rows = []
    for key_value, group in combined.groupby(key, sort=True, dropna=False):
        row = {key: key_value}
        for column in combined.columns:
            if column == key:
                continue
            values = _non_missing_values(group[column])
            if len(values) > 1:
                task2_conflict_rows.append({
                    "target_table": table,
                    "primary_key": key,
                    "key_value": key_value,
                    "target_field": column,
                    "non_missing_values": " | ".join(map(str, values)),
                })
                # Do not turn frame order into an accidental JSON-over-XML
                # precedence rule.  The unresolved value remains absent and is
                # carried as a visible FAIL in the validation register.
                row[column] = None
            else:
                row[column] = values[0] if values else None
        rows.append(row)
    return pd.DataFrame(rows, columns=combined.columns)


**Possible issue to review:** `_non_missing_values()` compares values with `==`, which is an
exact comparison for floats. Two exports that reach the same monetary amount by different
parsing routes (for example `money_series("AUD 5,287.78")` versus a native JSON float) could
in principle differ in the last binary digit and be reported as a conflict rather than
collapsing to one value. This does not occur on the current package — `task2_conflicts` is
empty — but it would be worth confirming with the tutor whether monetary comparison should use
`MONEY_TOLERANCE` here as it does in the Task 4 arithmetic checks. Note that a single shared
tolerance would be too coarse for `customer_lat`/`customer_long`, so any change should be
applied per field group rather than globally.


##### Reconcile the four overlapping source entities

Only these four target tables appear in both exports and therefore require reconciliation.
`customers` (JSON only) and `products` (XML only) are already unique on their keys and are
built directly from their single source in `T2-B1`.

Each call returns a frame at the target grain — one row per published key — which the build
sections below then type-convert and enrich.


In [ ]:
canonical_orders_source = reconcile_canonical("orders")
canonical_items_source = reconcile_canonical("order_items")
canonical_deliveries_source = reconcile_canonical("deliveries")
canonical_reviews_source = reconcile_canonical("product_reviews")


The check below answers one question: did the repeated and overlapping source records actually
collapse to the intended one-row-per-key grain? `source_rows` counts every normalised row read
from both exports, so it is expected to exceed `canonical_rows`. Within each canonical frame,
`canonical_rows` and `unique_keys` should agree and `duplicated_keys` should be zero.


In [ ]:
canonical_frames = {
    "orders": canonical_orders_source,
    "order_items": canonical_items_source,
    "deliveries": canonical_deliveries_source,
    "product_reviews": canonical_reviews_source,
}

canonical_grain_check = pd.DataFrame([
    {
        "target_table": table,
        "primary_key": PRIMARY_KEYS[table],
        "source_rows": sum(len(NORMALISED[(table, source)])
                           for source in ("JSON", "XML") if (table, source) in NORMALISED),
        "canonical_rows": len(frame),
        "unique_keys": frame[PRIMARY_KEYS[table]].nunique(dropna=False),
        "duplicated_keys": int(frame[PRIMARY_KEYS[table]].duplicated().sum()),
        "missing_keys": int(frame[PRIMARY_KEYS[table]].isna().sum()),
    }
    for table, frame in canonical_frames.items()
])
display(canonical_grain_check)


##### Narrative-field reconciliation

Narrative-derived fields cannot go through `reconcile_canonical()`, because the published
processing order requires the embedded references to be extracted from the **raw** parser
output — before HTML entity decoding and the rest of the cleaning pipeline. Reconciling the
normalised frame first would mean extracting from an already-modified string.

`reconcile_narrative_derivation()` therefore applies the Task 3 transform to each raw source
column, and only then reconciles the *derived* results by key, using the same
no-precedence rule as `reconcile_canonical()`.

One deliberate difference from `reconcile_canonical()` is worth noting: when no source
supplies a value the result is the literal `MISSING` sentinel required by the string contract,
whereas an unresolved **conflict** yields `None` so that it surfaces as a visible failure
rather than being disguised as ordinary absence.

The Task 3 text functions themselves are not modified here; Task 2 only consumes their output.


In [ ]:
def reconcile_narrative_derivation(table, target_field, source_specs, transform):
    '''Derive a narrative field from parser-obtained raw values, then reconcile it.

    This preserves the published processing order: reference extraction happens
    on the raw JSON value/XML element content, before entity decoding or other
    narrative cleaning.  Differing derived results are recorded and left
    unresolved rather than selected by source order.
    '''
    key = PRIMARY_KEYS[table]
    pieces = []
    for frame, source_key, source_value in source_specs:
        pieces.append(pd.DataFrame({
            key: norm_id(frame[source_key]),
            target_field: frame[source_value].map(transform),
        }))
    combined = pd.concat(pieces, ignore_index=True)
    reconciled = {}
    for key_value, group in combined.groupby(key, sort=True, dropna=False):
        values = _non_missing_values(group[target_field])
        if len(values) > 1:
            task2_conflict_rows.append({
                "target_table": table,
                "primary_key": key,
                "key_value": key_value,
                "target_field": target_field,
                "non_missing_values": " | ".join(map(str, values)),
            })
            reconciled[key_value] = None
        else:
            reconciled[key_value] = values[0] if values else MISSING
    return pd.Series(reconciled, name=target_field)


##### Derive the cleaned order and review narrative fields

Five derived series are produced, each keyed by the primary key of its target table so the
build sections can attach them with a simple `.map()`:

* `order_note_clean_by_id` / `order_promo_by_id` — from the raw order `customerNote`
  (`Customer_Note` in XML);
* `review_clean_by_id`, `review_order_reference_by_id`, `review_product_sku_by_id` — from the
  raw `reviewText` (`Review_Text` in XML).

Both note-derived series read the same source column but apply different transforms: one
cleans the prose for publication, the other extracts the promotion code before that cleaning
would have removed it.


In [ ]:
order_note_clean_by_id = reconcile_narrative_derivation(
    "orders", "customer_note_clean",
    [(json_headers, "orderID", "customerNote"),
     (xml_headers, "Order_ID", "Customer_Note")],
    clean_narrative_text,
)
order_promo_by_id = reconcile_narrative_derivation(
    "orders", "promo_code",
    [(json_headers, "orderID", "customerNote"),
     (xml_headers, "Order_ID", "Customer_Note")],
    extract_promo_code,
)
review_clean_by_id = reconcile_narrative_derivation(
    "product_reviews", "review_body_clean",
    [(json_reviews, "reviewID", "reviewText"),
     (xml_reviews, "Review_ID", "Review_Text")],
    clean_narrative_text,
)
review_order_reference_by_id = reconcile_narrative_derivation(
    "product_reviews", "extracted_order_reference",
    [(json_reviews, "reviewID", "reviewText"),
     (xml_reviews, "Review_ID", "Review_Text")],
    extract_order_reference,
)
review_product_sku_by_id = reconcile_narrative_derivation(
    "product_reviews", "extracted_product_sku",
    [(json_reviews, "reviewID", "reviewText"),
     (xml_reviews, "Review_ID", "Review_Text")],
    extract_product_sku,
)


A small sample makes the derivation behaviour visible: the cleaned prose should carry no
markup or markers, and the extracted reference/SKU columns should either hold a well-formed
code or the literal `NaN` sentinel. Rows are taken deterministically from the head of the
series so the output does not change between runs.


In [ ]:
narrative_sample = pd.DataFrame({
    "review_body_clean": review_clean_by_id,
    "extracted_order_reference": review_order_reference_by_id,
    "extracted_product_sku": review_product_sku_by_id,
}).head(5)
narrative_sample["review_body_clean"] = narrative_sample["review_body_clean"].str.slice(0, 60)
display(narrative_sample)

promo_sample = pd.DataFrame({
    "customer_note_clean": order_note_clean_by_id.str.slice(0, 60),
    "promo_code": order_promo_by_id,
}).head(5)
display(promo_sample)


##### Record every unresolved reconciliation conflict

Both reconciliation helpers append to the same `task2_conflict_rows` list, so this single
table is the complete conflict evidence for Task 2. It is written to the profiling directory
whether or not it contains rows: an empty file produced by the code path that *would* record a
conflict is stronger evidence than no file at all.

A genuine conflict is reported rather than raised, so the notebook continues, the six outputs
remain inspectable, and Task 4's `VAL-CONFLICT-01` carries the failure into the formal
register.


In [ ]:
task2_conflicts = pd.DataFrame(task2_conflict_rows, columns=[
    "target_table", "primary_key", "key_value", "target_field", "non_missing_values"
])
task2_conflict_path = os.path.join(PROFILE_DIR, f"{GROUP_ID}_T2_reconciliation_conflicts.csv")
task2_conflicts.to_csv(task2_conflict_path, index=False)
print(f"[saved] {task2_conflict_path}  ({len(task2_conflicts)} rows)")
if task2_conflicts.empty:
    print("Reconciliation: no field-level conflict detected after normalisation.")
else:
    # A genuine conflict is reported evidence, not a crash: the run continues so
    # the register and the six outputs stay inspectable. reconcile_canonical()
    # has already logged every disagreeing value to the profile above, and
    # VAL-CONFLICT-01 below carries the FAIL into the validation register.
    print(f"Reconciliation: {len(task2_conflicts)} field-level conflict(s) recorded in "
          f"{os.path.basename(task2_conflict_path)}; see check VAL-CONFLICT-01.")
    print(task2_conflicts.head(20).to_string(index=False))


##### Shared string helper

`trim_series()` is used by every table below. It collapses internal whitespace and optionally
applies the case required by the published categorical contract, while leaving genuine missing
values as `None` rather than converting them to an empty string. Keeping absence distinct from
an empty string matters because several target fields must later be written as the literal
`NaN` sentinel rather than as a blank CSV cell.


In [ ]:
def trim_series(series, case=None):
    '''Trim/collapse structured strings while retaining genuine missing values.'''
    def clean(value):
        if is_absent(value):
            return None
        result = " ".join(str(value).split())
        if case == "upper":
            return result.upper()
        if case == "lower":
            return result.lower()
        return result
    return series.map(clean)


#### `T2-B2` — Order items and published order arithmetic


##### Grain: one row per order item — and why this table is built first

`order_items` sits at the finest transactional grain in the model: an order may own many item
rows, so the item table is deliberately allowed to repeat `order_id` while the `orders` table
must not repeat it. Keeping the two grains separate is what stops the one-to-many relationship
from inflating the order-level table.

This section runs **before** the `orders` table is completed because the published arithmetic
flows upward from the item grain:

```
line_revenue  = round(quantity * unit_price, 2)          <- item grain (here)
order_price   = sum of rounded line_revenue by order_id  <- aggregated here
```

`orders.order_price` is therefore an aggregate of values that do not exist until this cell has
run. The section numbering below lists `orders` as 4.1 for readability, but the execution order
is dictated by this dependency and must not be rearranged.


###### Copy the canonical frame and standardise the identifier fields

The three identifiers are the join keys used by `orders`, `products` and `product_reviews`, so
they are trimmed to a single canonical spelling before anything is aggregated or matched on
them. Working on a copy leaves `canonical_items_source` intact for the reconciliation
diagnostics.


In [ ]:
order_items = canonical_items_source.copy()
order_items["order_item_id"] = trim_series(order_items["order_item_id"])
order_items["order_id"] = trim_series(order_items["order_id"])
order_items["product_id"] = trim_series(order_items["product_id"])


###### Convert the measures to their published types

The target schema represents `quantity` as a whole-number measure and `unit_price` as a
monetary value in cents. Converting after reconciliation rather than before ensures the
equivalent JSON and XML representations have already been collapsed to one value, so the cast
is applied once to a canonical value instead of twice to two different notations.


In [ ]:
order_items["quantity"] = order_items["quantity"].astype(int)
order_items["unit_price"] = round2(order_items["unit_price"].astype(float))


###### Step 1 of the published arithmetic — `line_revenue`

`line_revenue` is re-derived rather than trusted from the source, so the submitted value is
reproducible from its own components. Rounding happens **at the line level**, before any
aggregation, because the published `order_price` is the sum of already-rounded line values;
summing unrounded products and rounding once at the end would give a different answer on some
orders. `round2` is Python's decimal rounding rather than numpy's, per `ASM-14`.


In [ ]:
order_items["line_revenue"] = round2(order_items["quantity"] * order_items["unit_price"])


###### Step 2 — aggregate the item grain up to the order grain

`order_price_by_id` moves from the item grain to the order grain. It is keyed by `order_id` so
the `orders` table can attach it with `.map()`. This series is the only reason `order_items`
must be built before `orders` is finished.


In [ ]:
order_price_by_id = round2(order_items.groupby("order_id", sort=True)["line_revenue"].sum())


The diagnostics below answer three questions: did the type conversions produce the expected
dtypes, is the item primary key still complete and unique, and does the step-1 arithmetic hold
on a readable sample of rows?


In [ ]:
print(f"order_items shape: {order_items.shape}")
print(f"order_price_by_id: {len(order_price_by_id):,} orders")

display(order_items[["quantity", "unit_price", "line_revenue"]].dtypes.rename("dtype").to_frame())

print(f"missing order_item_id    : {int(order_items['order_item_id'].isna().sum())}")
print(f"duplicated order_item_id : {int(order_items['order_item_id'].duplicated().sum())}")

arithmetic_spot_check = order_items[
    ["order_item_id", "order_id", "product_id", "quantity", "unit_price", "line_revenue"]
].head()
display(arithmetic_spot_check)


If the duplicated-key count above is zero and the unique key count matches the row count, the
intended one-row-per-order-item grain has been preserved through reconciliation. The sample
rows let the step-1 arithmetic be verified by eye before it is aggregated into `order_price`.


### 4.1 `orders`


##### Grain: one row per canonical order

Each row is one order, identified by `order_id`. The order-level monetary fields are re-derived
from the item grain built above, so this table must not contain repeated `order_id` values —
a duplicate here would double-count revenue against the same customer.

The build proceeds in stages: identifiers and categories, then numeric fields, then the
narrative fields reconciled in `T2-R1`, then the published arithmetic, and finally the removal
of the source-only column.


###### Initialise from the canonical frame and standardise structured strings

`currency` and `coupon_code` are upper-cased because the published categorical contract stores
them that way; `coupon_code` additionally takes the literal `NaN` sentinel where the order
carries no coupon, since the dictionary specifies a string result rather than a blank cell.


In [ ]:
orders = canonical_orders_source.copy()
orders["order_id"] = trim_series(orders["order_id"])
orders["source_system_record_id"] = trim_series(orders["source_system_record_id"])
orders["customer_id"] = trim_series(orders["customer_id"])
orders["currency"] = trim_series(orders["currency"], "upper")
orders["coupon_code"] = trim_series(orders["coupon_code"], "upper").fillna(MISSING)


###### Standardise `coupon_discount` and the remaining numeric fields

`coupon_discount` is a percentage compared exactly rather than on a tolerance, so its written
representation is part of the contract. The value is parsed as a float and then narrowed back
to an integer only when every discount in the package is a whole percentage point — a decision
taken from the data rather than asserted, so a future package carrying a fractional discount
would keep its precision automatically.


In [ ]:
# coupon_discount is compared "exact after published normalisation", not on a
# tolerance, so the written representation matters. Both exports publish whole
# percentage points ("15" / "15%"), and every other integral number in the six
# outputs is written as an integer; writing "15.0" here would be the only float
# stand-in for an integer. Derived from the data, not asserted: keep the float
# parse if this package ever carries a fractional discount.
coupon_discount = orders["coupon_discount"].astype(float)
orders["coupon_discount"] = (
    coupon_discount.astype(int) if (coupon_discount % 1 == 0).all() else coupon_discount
)
orders["delivery_charges"] = round2(orders["delivery_charges"].astype(float))
orders["customer_lat"] = orders["customer_lat"].astype(float)
orders["customer_long"] = orders["customer_long"].astype(float)


###### Attach the reconciled narrative and promotion values

Both series were derived in `T2-R1` from the raw order note and are keyed by `order_id`, so
they attach cleanly at this grain. The promotion code had to be extracted before the note was
cleaned, which is why the derivation lives in the reconciliation section rather than here.


In [ ]:
orders["customer_note_clean"] = orders["order_id"].map(order_note_clean_by_id)
orders["promo_code"] = orders["order_id"].map(order_promo_by_id)


###### Attach `order_price` from the item grain

This is where the dependency on `T2-B2` is realised. Every order is expected to own at least
one canonical item line, so a missing `order_price` would indicate that reconciliation lost
item rows rather than that the order genuinely has no value.


In [ ]:
orders["order_price"] = orders["order_id"].map(order_price_by_id)
assert orders["order_price"].notna().all(), "an order has no canonical order-item lines"


**Possible issue to review:** this `assert` halts the entire notebook if any order lacks item
lines, which conflicts with the approach taken everywhere else in Task 2, where a problem is
recorded as visible evidence and the run continues so the outputs and the register remain
inspectable. If it ever fired, no CSVs and no validation register would be produced — the
marker would see a traceback instead of the diagnosis. It currently passes, so the behaviour is
left unchanged, but converting it into a `VAL-*` check (or a printed warning plus a recorded
conflict row) would be more consistent. Worth confirming the expected behaviour with the tutor.


###### Steps 3 to 6 of the published arithmetic

The remaining published steps run in a fixed sequence:

```
tax_amount              = round(order_price / 11, 2)
discounted order price  = order_price * (1 - coupon_discount / 100)
order_total             = round(discounted order price + delivery_charges, 2)
```

`tax_amount` is **included GST**: it is already contained within `order_price` and is reported
separately for transparency. It must **not** be added again when forming `order_total`, and it
is computed from the undiscounted `order_price` rather than from the discounted figure.

`order_total` applies the discount to the goods value first and then adds `delivery_charges`,
so delivery is not discounted. Rounding is applied once, to the final sum.


In [ ]:
orders["tax_amount"] = round2(orders["order_price"] / 11)
orders["order_total"] = round2(
    orders["order_price"] * (1 - orders["coupon_discount"] / 100)
    + orders["delivery_charges"]
)


###### Remove the source-only column

`customer_note` is the raw narrative that `customer_note_clean` and `promo_code` were derived
from. It is not a published field, so it is dropped here rather than being allowed to reach
the schema-enforcement step in `T2-O1`.


In [ ]:
orders = orders.drop(columns=["customer_note"])


The sample below shows the arithmetic components alongside their results, so the discount and
included-GST behaviour can be checked on a handful of orders without printing the whole table.


In [ ]:
print(f"orders shape: {orders.shape}")
print(f"missing order_id    : {int(orders['order_id'].isna().sum())}")
print(f"duplicated order_id : {int(orders['order_id'].duplicated().sum())}")

display(orders[[
    "order_id", "order_price", "coupon_discount", "delivery_charges",
    "tax_amount", "order_total",
]].head())


Reading across a row: `tax_amount` is a fraction of `order_price` and is **not** part of the
`order_total` sum, while `order_total` reflects the discounted goods value plus the undiscounted
delivery charge. A duplicated-key count of zero confirms the one-row-per-order grain survived
the mapping and aggregation steps.


### 4.2 `order_items`


`order_items` is built in `T2-B2` above rather than here, because the published arithmetic
runs in the opposite direction to the section order: `orders.order_price` is the sum of the
rounded `line_revenue` values, so the item grain has to be materialised before the order grain
can be completed. Building it inside this section would leave section 4.1 referring to a table
that does not exist yet.

The finished table is shown below at its published grain and field order.


In [ ]:
# The dictionary field order is applied to every table together in section 7; here the
# item grain is simply shown as built, in the published field order for readability.
order_items_fields = list(dictionary.loc[dictionary.output_table == "order_items", "field_name"])
order_items_view = order_items[order_items_fields]

print(f"order_items: {len(order_items_view):,} rows x {order_items_view.shape[1]} columns")
print(f"  order_item_id complete and unique : "
      f"{order_items_view['order_item_id'].is_unique and order_items_view['order_item_id'].notna().all()}")
print(f"  line_revenue == round(quantity * unit_price, 2) : "
      f"{int((order_items_view['line_revenue'] == round2(order_items_view['quantity'] * order_items_view['unit_price'])).sum()):,}"
      f"/{len(order_items_view):,}")
display(order_items_view.head())


### 4.3 `customers`


#### `T2-B1` — Build the two single-source entity tables


##### Grain: one row per customer

`customers` and `products` are the two tables that appear in only one export, so neither needs
`reconcile_canonical()`. `T1-P3` established that both are already unique on their business
key within their source, which is why they can be constructed directly.

`customers` is JSON-only and is assembled in a single `pd.DataFrame({...})` because each target
field maps to exactly one source field through one transformation. The transformations fall
into four groups:

* **identifiers and categoricals** — `trim_series()`, with case applied where the published
  contract fixes it (`home_state` upper; `preferred_language` and `email_domain` lower);
* **dates** — `signupDate` is already ISO in this export, so it is parsed and re-formatted to
  guarantee the output format rather than trusting the input string;
* **numeric measures** — `prior_12m_orders` as a whole-number count, and
  `lifetime_value_before_period` as a rounded monetary value;
* **booleans** — `marketing_consent`, which JSON already publishes as a native boolean.


In [ ]:
customers = pd.DataFrame({
    "customer_id": trim_series(json_customers["customerID"]),
    "signup_date": pd.to_datetime(json_customers["signupDate"], format="%Y-%m-%d").dt.strftime("%Y-%m-%d"),
    "loyalty_tier": trim_series(json_customers["loyaltyTier"]),
    "customer_segment": trim_series(json_customers["customerSegment"]),
    "age_band": trim_series(json_customers["ageBand"]),
    "preferred_channel": trim_series(json_customers["preferredChannel"]),
    "home_suburb": trim_series(json_customers["homeSuburb"]),
    "prior_12m_orders": json_customers["prior12MOrders"].astype(int),
    "lifetime_value_before_period": round2(json_customers["lifetimeValueBeforePeriod"].astype(float)),
    "marketing_consent": json_customers["marketingConsent"].astype(bool),
    "home_postcode": trim_series(json_customers["homePostcode"]),
    "home_state": trim_series(json_customers["homeState"], "upper"),
    "home_country": trim_series(json_customers["homeCountry"]),
    "preferred_language": trim_series(json_customers["preferredLanguage"], "lower"),
    "acquisition_source": trim_series(json_customers["acquisitionSource"]),
    "account_status": trim_series(json_customers["accountStatus"]),
    "preferred_device": trim_series(json_customers["preferredDevice"]),
    "email_domain": trim_series(json_customers["emailDomain"], "lower"),
    "household_size_band": trim_series(json_customers["householdSizeBand"]),
    "contact_frequency_preference": trim_series(json_customers["contactFrequencyPreference"]),
})


The customer primary key is checked here, and `home_state` is shown because it is one of the
fields where casing was deliberately changed — the category list confirms the upper-casing
merged notational variants rather than creating new categories.


In [ ]:
print(f"customers shape: {customers.shape}")
print(f"missing customer_id    : {int(customers['customer_id'].isna().sum())}")
print(f"duplicated customer_id : {int(customers['customer_id'].duplicated().sum())}")

display(customers[["prior_12m_orders", "lifetime_value_before_period",
                   "marketing_consent", "signup_date"]].dtypes.rename("dtype").to_frame())

display(customers["home_state"].value_counts(dropna=False).to_frame("rows"))
display(customers.head())


### 4.4 `deliveries`


#### `T2-B3` — Completed-order deliveries and canonical reviews


##### Grain: one row per completed order delivery

The published grain for this table is not "one row per delivery record in the source" but one
row per delivery **of a completed order**. The source exports carry delivery blocks for orders
in other states as well, so the canonical frame is filtered by the set of completed
`order_id` values taken from the finished `orders` table.

Deriving `completed_order_ids` from `orders` rather than from the raw source guarantees the
filter uses the same normalised `order_status` and the same trimmed `order_id` spelling that
the published order table uses, so the two tables cannot disagree about which orders are
complete.


In [ ]:
completed_order_ids = set(orders.loc[orders["order_status"] == "Completed", "order_id"])
deliveries = canonical_deliveries_source[
    canonical_deliveries_source["order_id"].isin(completed_order_ids)
].copy()


**Possible issue to review:** `"Completed"` is written here as a literal. Everywhere else in the
notebook a categorical domain is taken from the measured `CATEGORICAL_DOMAINS` built in `T1-P4`
rather than hard-coded. If a future package spelled the status differently (`COMPLETED`, or
with trailing whitespace), this filter would silently match nothing, `deliveries` would be
empty, and the Task 4 grain check would compare two empty sets and still report agreement. It
is correct for the current package, but worth asking the tutor whether the status literal should
be validated against the measured domain before it is used as a filter.


###### Standardise the structured string fields

These are the delivery identifiers and the operational categoricals. They are trimmed but not
case-folded, because the published capitalisation of the carrier and status values is itself
part of the categorical contract measured in `T1-P4`.


In [ ]:
for column in ("delivery_id", "order_id", "carrier", "service_level", "delivery_status",
               "delay_reason", "delivery_window"):
    deliveries[column] = trim_series(deliveries[column])


###### Convert the operational and monetary measures

The four counters are whole-number measures; `delivery_cost` is monetary and is rounded to
cents; `shipping_distance_km` and `estimated_carbon_kg` are continuous physical measures and
keep their full precision rather than being rounded to a monetary scale.


In [ ]:
for column in ("delay_days", "fulfilment_hours", "promised_days", "tracking_event_count"):
    deliveries[column] = deliveries[column].astype(int)
deliveries["delivery_cost"] = round2(deliveries["delivery_cost"].astype(float))
for column in ("shipping_distance_km", "estimated_carbon_kg"):
    deliveries[column] = deliveries[column].astype(float)


###### Handle the delivery note

Unlike the order note and the review body, the delivery note arrives already clean in both
exports, so it needs no narrative cleaning pipeline — only trimming, and the literal `NaN`
sentinel where the note is absent.


In [ ]:
# This field is already clean and its published case is part of the categorical contract.
deliveries["delivery_note_clean"] = trim_series(deliveries["delivery_note_clean"]).fillna(MISSING)


The delivery primary key is checked below, together with a sample of the operational fields.
Because the table was filtered rather than aggregated, `order_id` should also be unique here —
a repeated `order_id` would mean one completed order carried more than one delivery record.


In [ ]:
print(f"deliveries shape: {deliveries.shape}")
print(f"completed orders in `orders`   : {len(completed_order_ids):,}")
print(f"missing delivery_id            : {int(deliveries['delivery_id'].isna().sum())}")
print(f"duplicated delivery_id         : {int(deliveries['delivery_id'].duplicated().sum())}")
print(f"duplicated order_id            : {int(deliveries['order_id'].duplicated().sum())}")

display(deliveries[[
    "delivery_id", "order_id", "carrier", "service_level",
    "delivery_status", "delay_days", "delivery_cost",
]].head())


### 4.5 `products`


##### Grain: one row per product

`products` is the second single-source table (XML only) and, like `customers`, is already
unique on its key in the source. Because `read_xml` was given `dtype=str` in section 1, every
value arrives as text and each target field therefore needs an explicit conversion:

* **monetary** — `unit_price` and `unit_cost` pass through `money_series()` to strip the
  currency prefix and thousands separators profiled in `T1-P4` before rounding to cents;
* **dates** — `Launch_Date` is day-first in this export (confirmed by measurement in `T1-P4`,
  not assumed), so it is parsed with `format="%d/%m/%Y"` and re-emitted as ISO;
* **booleans** — `norm_bool()` maps the XML `Y`/`N` spelling to real booleans;
* **SKU casing** — `product_sku` is upper-cased so it can be compared exactly against the SKU
  extracted from review narrative in Task 4;
* **narrative** — `product_description_clean` is the only field here needing the Task 3
  cleaning pipeline. It is derived directly rather than through
  `reconcile_narrative_derivation()` because there is no second source to reconcile against.


In [ ]:
products = pd.DataFrame({
    "product_id": trim_series(xml_products["Product_ID"]),
    "product_name": trim_series(xml_products["Product_Name"]),
    "category": trim_series(xml_products["Category"]),
    "brand": trim_series(xml_products["Brand"]),
    "unit_price": round2(money_series(xml_products["Unit_Price"])),
    "unit_cost": round2(money_series(xml_products["Unit_Cost"])),
    "launch_year": xml_products["Launch_Year"].astype(int),
    "warranty_months": xml_products["Warranty_Months"].astype(int),
    "weight_kg": xml_products["Weight_Kg"].astype(float),
    "product_sku": trim_series(xml_products["Product_Sku"], "upper"),
    "subcategory": trim_series(xml_products["Subcategory"]),
    "model_family": trim_series(xml_products["Model_Family"]),
    "colour": trim_series(xml_products["Colour"]),
    "supplier_id": trim_series(xml_products["Supplier_ID"]),
    "supplier_country": trim_series(xml_products["Supplier_Country"]),
    "launch_date": pd.to_datetime(xml_products["Launch_Date"], format="%d/%m/%Y").dt.strftime("%Y-%m-%d"),
    "tax_category": trim_series(xml_products["Tax_Category"], "upper"),
    "package_type": trim_series(xml_products["Package_Type"]),
    "recyclable_packaging": norm_bool(xml_products["Recyclable_Packaging"]),
    "active_flag": norm_bool(xml_products["Active_Flag"]),
    "product_description_clean": xml_products["Product_Description"].map(clean_narrative_text),
})


The dtype check below is the useful one for this table: because every source value was read as
text, it confirms the monetary, numeric, date and boolean conversions all took effect rather
than leaving strings in the output.


In [ ]:
print(f"products shape: {products.shape}")
print(f"missing product_id    : {int(products['product_id'].isna().sum())}")
print(f"duplicated product_id : {int(products['product_id'].duplicated().sum())}")

display(products[[
    "unit_price", "unit_cost", "launch_year", "weight_kg",
    "launch_date", "recyclable_packaging", "active_flag",
]].dtypes.rename("dtype").to_frame())

display(products[[
    "product_id", "product_sku", "category", "unit_price", "unit_cost", "launch_date",
]].head())


### 4.6 `product_reviews`


##### Grain: one row per canonical product review

Each row is one review, identified by `review_id`. A review points at one order, one order
item, one product and one customer, so this table sits on the many side of four separate
relationships; those foreign keys come from the structured source and are never inferred from
the review text.

This is the most involved table in Task 2 because it combines structured fields with five
narrative-derived fields reconciled in `T2-R1` and two measures computed from the cleaned text.


###### Initialise and standardise the identifier fields

All five identifiers are trimmed to their canonical spelling so the foreign-key checks later in
the notebook compare like with like.


In [ ]:
product_reviews = canonical_reviews_source.copy()
for column in ("review_id", "order_id", "order_item_id", "product_id", "customer_id"):
    product_reviews[column] = trim_series(product_reviews[column])


###### Standardise the language, rating, title and vote fields

`language_code` is lower-cased to match the published contract. `rating` and `helpful_votes`
are whole-number measures. `review_title` takes the literal `NaN` sentinel when absent, because
the dictionary specifies a string result for this field rather than a blank cell.


In [ ]:
product_reviews["language_code"] = trim_series(product_reviews["language_code"], "lower")
product_reviews["rating"] = product_reviews["rating"].astype(int)
product_reviews["review_title"] = trim_series(product_reviews["review_title"]).fillna(MISSING)
product_reviews["helpful_votes"] = product_reviews["helpful_votes"].astype(int)


###### Attach the cleaned review body and derive the script fields

`review_body_clean` is the reconciled output of the Task 3 cleaning pipeline. The two script
fields are then derived **from `review_body_clean`, not from the raw review text**: the raw
value still contains markup, URLs and marker tokens, so a Latin-script analysis computed from
it would be describing the noise rather than the human-readable review. Deriving them from the
cleaned field also guarantees the indicator and the analysis field describe the same string
that is actually published.


In [ ]:
product_reviews["review_body_clean"] = product_reviews["review_id"].map(review_clean_by_id)
product_reviews["review_body_latin_analysis"] = product_reviews["review_body_clean"].map(build_latin_analysis)
product_reviews["contains_non_latin_script"] = product_reviews["review_body_clean"].map(contains_non_latin_script)


###### Attach the extracted order and SKU references

Both were extracted in `T2-R1` from the **raw** review text, before cleaning could remove the
reference wrappers. They are attached here by `review_id`. Task 4 later confirms that each
extracted value agrees with the structured foreign key, which is the real test of the
extraction: a reference that disagrees with `order_id` or `product_id` indicates a boundary
problem in the pattern rather than something to be repaired by overwriting either side.


In [ ]:
product_reviews["extracted_order_reference"] = product_reviews["review_id"].map(
    review_order_reference_by_id
)
product_reviews["extracted_product_sku"] = product_reviews["review_id"].map(
    review_product_sku_by_id
)


###### Compute the review measures, with explicit sentinel handling

Both measures describe the cleaned, human-readable review body. Where the body is the literal
`NaN` sentinel, the measure is defined as `0` rather than `3` — the sentinel is a marker of
absence, not three characters of review text, and counting its characters would report a
review that does not exist as though it had content.


In [ ]:
product_reviews["review_length_chars"] = product_reviews["review_body_clean"].map(
    lambda value: 0 if value == MISSING else len(value)
)
product_reviews["review_word_count"] = product_reviews["review_body_clean"].map(
    lambda value: 0 if value == MISSING else len(value.split())
)


###### Remove the source-only column

`review_text` is the raw narrative that five published fields were derived from. Like
`orders.customer_note` it is not itself a published field and is dropped before schema
enforcement.


In [ ]:
product_reviews = product_reviews.drop(columns=["review_text"])


The sample below places the cleaned body next to its Latin-only analysis and the derived
measures, so the multilingual behaviour is visible: the clean field should retain non-Latin
letters while the analysis field should not, and the indicator should agree with the clean
field. The body columns are truncated for display only.


In [ ]:
print(f"product_reviews shape: {product_reviews.shape}")
print(f"missing review_id    : {int(product_reviews['review_id'].isna().sum())}")
print(f"duplicated review_id : {int(product_reviews['review_id'].duplicated().sum())}")

review_sample = product_reviews[[
    "review_id", "language_code", "rating", "review_body_clean",
    "review_body_latin_analysis", "contains_non_latin_script",
    "review_length_chars", "review_word_count",
]].head(5).copy()
review_sample["review_body_clean"] = review_sample["review_body_clean"].str.slice(0, 45)
review_sample["review_body_latin_analysis"] = review_sample["review_body_latin_analysis"].str.slice(0, 45)
display(review_sample)


#### `T2-O1` — Enforce the public schema and write exactly six outputs


##### Schema contract

The six in-memory tables are now complete, but they are not yet submittable. Three rules apply
before export:

* **exact fields** — each table must contain precisely the fields the public dictionary lists
  for it, with no field missing and no extra column;
* **no helper columns** — any source-only or intermediate column must already have been
  dropped; the mismatch check below is what makes an accidental survivor fail loudly rather
  than reach the CSV;
* **dictionary field order** — the dictionary, not the construction order, controls the column
  order in the output.

Rows are then sorted by primary key with a stable sort, so re-running the notebook produces
byte-identical files.


In [ ]:
STANDARDISED = {
    "orders": orders,
    "order_items": order_items,
    "customers": customers,
    "deliveries": deliveries,
    "products": products,
    "product_reviews": product_reviews,
}

TASK2_PRIMARY_KEYS = {
    "orders": "order_id",
    "order_items": "order_item_id",
    "customers": "customer_id",
    "deliveries": "delivery_id",
    "products": "product_id",
    "product_reviews": "review_id",
}


###### Apply the dictionary field order and deterministic row order

The mismatch check compares the built columns against the dictionary in both directions, so a
missing target field and a leftover helper column are both caught. Reindexing with
`frame[required_columns]` then imposes the published order.


In [ ]:
for table, frame in STANDARDISED.items():
    required_columns = list(dictionary.loc[dictionary.output_table == table, "field_name"])
    missing_columns = sorted(set(required_columns) - set(frame.columns))
    extra_columns = sorted(set(frame.columns) - set(required_columns))
    assert not missing_columns and not extra_columns, (
        f"{table} schema mismatch: missing={missing_columns}, extra={extra_columns}"
    )
    STANDARDISED[table] = (frame[required_columns]
                           .sort_values(TASK2_PRIMARY_KEYS[table], kind="stable")
                           .reset_index(drop=True))


###### Write the six CSV files

Filenames follow the required `<group>_<table>_standardised.csv` convention and are built from
the configured `OUTPUT_DIR` and `GROUP_ID`, so no absolute path is embedded in the notebook.
The files are written here, before the validation register runs, so that Task 4 can validate
the serialized deliverables rather than only the in-memory frames.


In [ ]:
OUTPUT_PATHS = {
    table: os.path.join(OUTPUT_DIR, f"{GROUP_ID}_{table}_standardised.csv")
    for table in STANDARDISED
}

for table, frame in STANDARDISED.items():
    frame.to_csv(OUTPUT_PATHS[table], index=False)
    print(f"[saved] {OUTPUT_PATHS[table]}  ({len(frame):,} rows x {frame.shape[1]} columns)")


##### Final shape and primary-key summary

One row per submitted table, computed from the exported frames themselves. `rows` and
`unique_keys` should agree, and both `missing_pks` and `duplicated_pks` should be zero, for
every table.


In [ ]:
task2_table_summary = pd.DataFrame([
    {
        "table": table,
        "rows": len(frame),
        "columns": frame.shape[1],
        "primary_key": TASK2_PRIMARY_KEYS[table],
        "unique_keys": frame[TASK2_PRIMARY_KEYS[table]].nunique(dropna=False),
        "missing_pks": int(frame[TASK2_PRIMARY_KEYS[table]].isna().sum()),
        "duplicated_pks": int(frame[TASK2_PRIMARY_KEYS[table]].duplicated().sum()),
    }
    for table, frame in STANDARDISED.items()
])
display(task2_table_summary)


## 5. Reconcile overlap and verify relationships


##### Row flow from source to canonical grain

This table records how many normalised source rows were read for each reconciled entity and
how many canonical rows survived. `source_rows` counts both exports together, so it is expected
to be larger than `canonical_rows`: the difference is exactly the within-source repeats and the
cross-source overlap measured in `T1-P6`. `canonical_rows` and `canonical_primary_keys` should
match, confirming the target grain.


In [ ]:
task2_reconciliation = pd.DataFrame([
    {
        "target_table": table,
        "source_rows": sum(len(NORMALISED[(table, source)])
                           for source in ("JSON", "XML") if (table, source) in NORMALISED),
        "canonical_rows": len(frame),
        "canonical_primary_keys": frame[PRIMARY_KEYS[table]].nunique(dropna=False),
    }
    for table, frame in [
        ("orders", canonical_orders_source),
        ("order_items", canonical_items_source),
        ("deliveries", canonical_deliveries_source),
        ("product_reviews", canonical_reviews_source),
    ]
])
task2_reconciliation


##### Relationship sanity check

A lightweight look at the eight foreign keys across the six finished tables, reported as simple
unmatched counts. Each row anti-joins the child values against the parent key set, so
`unmatched_values` counts distinct child values with no parent, and `unmatched_rows` counts the
rows affected.

This is inspection only — it exists so a broken relationship is obvious while building. The
formal `VAL-FK-*` checks with PASS/FAIL status, observed evidence and resolutions are in Task 4.


In [ ]:
task2_fk_pairs = [
    ("orders", "customer_id", "customers", "customer_id"),
    ("order_items", "order_id", "orders", "order_id"),
    ("order_items", "product_id", "products", "product_id"),
    ("deliveries", "order_id", "orders", "order_id"),
    ("product_reviews", "order_id", "orders", "order_id"),
    ("product_reviews", "order_item_id", "order_items", "order_item_id"),
    ("product_reviews", "product_id", "products", "product_id"),
    ("product_reviews", "customer_id", "customers", "customer_id"),
]

task2_fk_overview = pd.DataFrame([
    {
        "child": f"{child_table}.{child_field}",
        "parent": f"{parent_table}.{parent_field}",
        "distinct_child_values": STANDARDISED[child_table][child_field].nunique(dropna=False),
        "unmatched_values": len(
            set(STANDARDISED[child_table][child_field])
            - set(STANDARDISED[parent_table][parent_field])
        ),
        "unmatched_rows": int((~STANDARDISED[child_table][child_field]
                               .isin(set(STANDARDISED[parent_table][parent_field]))).sum()),
    }
    for child_table, child_field, parent_table, parent_field in task2_fk_pairs
])
display(task2_fk_overview)


Every relationship listed above is expected to show zero unmatched values. A non-zero count
would point at one of three causes — a parent row lost during reconciliation, an identifier
normalised inconsistently between child and parent, or a broken flattening relationship in
section 1 — and should be traced rather than patched by dropping the offending child rows.


## 6. Validation register

#### Task 4 — HD validation register (`VAL-*`)

### 6.1 Schema and type checks (`VAL-SCHEMA-...`)

In [46]:
task2_checks = []

def task2_check(check_id, description, passed, observed, evidence,
                pass_interpretation, fail_resolution):
    '''Append one complete, citable validation-register row.'''
    status = "PASS" if bool(passed) else "FAIL"
    task2_checks.append({
        "check_id": check_id,
        "description": description,
        "observed": str(observed),
        "status": status,
        "evidence": evidence,
        "resolution_or_interpretation": (
            pass_interpretation if status == "PASS" else fail_resolution
        ),
    })

# Read both the typed and exact-text submitted representations.  The latter
# keeps identifier zeros, boolean spelling and the literal NaN sentinel visible.
written = {table: pd.read_csv(path, keep_default_na=False)
           for table, path in OUTPUT_PATHS.items()}
written_text = {table: pd.read_csv(path, dtype=str, keep_default_na=False)
                for table, path in OUTPUT_PATHS.items()}

expected_output_files = {os.path.basename(path) for path in OUTPUT_PATHS.values()}
observed_output_files = {os.path.basename(path) for path in glob.glob(os.path.join(OUTPUT_DIR, "*.csv"))}
task2_check(
    "VAL-FILE-01", "exactly six required output files",
    observed_output_files == expected_output_files, sorted(observed_output_files),
    "T2-O1; OUTPUT_PATHS compared with every CSV physically present in OUTPUT_DIR.",
    "All and only the six public data products are present.",
    "Remove extra CSVs or regenerate the missing/misnamed public data product.",
)

for table, frame in written_text.items():
    contract = dictionary[dictionary.output_table == table]
    expected_columns = list(contract.field_name)
    task2_check(
        f"VAL-SCHEMA-{table}", f"{table}: exact field names and order",
        list(frame.columns) == expected_columns, list(frame.columns),
        f"public_data_dictionary.csv positions compared with {os.path.basename(OUTPUT_PATHS[table])}.",
        "The submitted table has no missing, extra or reordered field.",
        "Reindex the table to the dictionary positions and remove helper columns before export.",
    )

    key = TASK2_PRIMARY_KEYS[table]
    pk_ok = frame[key].ne("").all() and frame[key].is_unique
    pk_observed = f"rows={len(frame)}, distinct={frame[key].nunique()}, blank={(frame[key] == '').sum()}"
    task2_check(
        f"VAL-PK-{table}", f"{table}: primary key complete and unique",
        pk_ok, pk_observed,
        f"Exact-text {key} values in the submitted CSV; grain from public_data_dictionary.csv.",
        "One complete primary key identifies every row at the required grain.",
        "Trace blank/repeated keys to within-source duplication or incorrect flattening, then reconcile by the published key.",
    )

    required_fields = list(contract.loc[~contract.nullable.astype(bool), "field_name"])
    missing_required = {
        column: int(frame[column].isna().sum() + frame[column].eq("").sum())
        for column in required_fields
        if frame[column].isna().any() or frame[column].eq("").any()
    }
    task2_check(
        f"VAL-MISSING-{table}", f"{table}: required fields have no empty/pandas missing values",
        not missing_required, missing_required or "none",
        "Submitted CSV read with dtype=str and keep_default_na=False, preserving empty and literal NaN values.",
        "All required fields are represented; prescribed string absence remains a visible literal NaN.",
        "Derive or reconcile the missing required value; use literal NaN only where the string contract prescribes it.",
    )

    # Validate what is serialized, not only the pre-export DataFrame dtype.
    type_errors = []
    for row in contract.itertuples():
        series = frame[row.field_name]
        if row.data_type == "string":
            valid = series.map(lambda value: isinstance(value, str)).all()
        elif row.data_type == "number":
            valid = series.ne("").all() and pd.to_numeric(series, errors="coerce").notna().all()
        elif row.data_type == "boolean":
            valid = series.ne("").all() and set(series) <= {"True", "False"}
        elif row.data_type == "date":
            parsed = pd.to_datetime(series, format="%Y-%m-%d", errors="coerce")
            valid = parsed.notna().all() and parsed.dt.strftime("%Y-%m-%d").eq(series).all()
        elif row.data_type == "datetime":
            parsed = pd.to_datetime(series, format="%Y-%m-%d %H:%M:%S", errors="coerce")
            valid = parsed.notna().all() and parsed.dt.strftime("%Y-%m-%d %H:%M:%S").eq(series).all()
        else:
            valid = False
        if not valid:
            type_errors.append(f"{row.field_name}:{row.data_type}")
    task2_check(
        f"VAL-TYPE-{table}", f"{table}: serialized types and date/time formats",
        not type_errors, type_errors or "all fields match",
        "Submitted CSV lexical values checked against every public dictionary data_type.",
        "Numbers parse, booleans are exactly True/False, and dates/timestamps round-trip in the required format.",
        "Correct the listed serialization or conversion before writing the submitted CSV.",
    )

# Allowed categorical values are data-derived from the parser outputs, never hard-coded.
for (table, field), allowed_values in CATEGORICAL_DOMAINS.items():
    observed_values = set(written_text[table][field])
    allowed = set(map(str, allowed_values))
    unexpected = sorted(observed_values - allowed)
    task2_check(
        f"VAL-DOMAIN-{table}-{field}", f"{table}.{field}: values stay in measured source domain",
        not unexpected, f"observed={sorted(observed_values)}, unexpected={unexpected}",
        f"T1-P4 measured source domain compared with exact-text submitted {table}.{field}.",
        "No structured category was invented, case-folded or lost during transformation.",
        "Trace unexpected values to an incorrect case/whitespace conversion or an unrecorded source category.",
    )

# Sensible ranges required by the HD validation descriptor.
range_contract = [
    ("orders", "coupon_discount", lambda s: s.between(0, 100), "0 to 100 percentage points"),
    ("orders", "customer_lat", lambda s: s.between(-90, 90), "valid latitude"),
    ("orders", "customer_long", lambda s: s.between(-180, 180), "valid longitude"),
    ("order_items", "quantity", lambda s: s.ge(1), "at least one unit"),
    ("order_items", "unit_price", lambda s: s.ge(0), "non-negative"),
    ("customers", "prior_12m_orders", lambda s: s.ge(0), "non-negative"),
    ("customers", "lifetime_value_before_period", lambda s: s.ge(0), "non-negative"),
    ("deliveries", "delay_days", lambda s: s.ge(0), "non-negative"),
    ("deliveries", "fulfilment_hours", lambda s: s.ge(0), "non-negative"),
    ("deliveries", "delivery_cost", lambda s: s.ge(0), "non-negative"),
    ("deliveries", "shipping_distance_km", lambda s: s.ge(0), "non-negative"),
    ("deliveries", "estimated_carbon_kg", lambda s: s.ge(0), "non-negative"),
    ("products", "unit_price", lambda s: s.ge(0), "non-negative"),
    ("products", "unit_cost", lambda s: s.ge(0), "non-negative"),
    ("products", "weight_kg", lambda s: s.gt(0), "positive"),
    ("product_reviews", "rating", lambda s: s.between(1, 5), "1 to 5"),
    ("product_reviews", "helpful_votes", lambda s: s.ge(0), "non-negative"),
    ("product_reviews", "review_length_chars", lambda s: s.ge(0), "non-negative"),
    ("product_reviews", "review_word_count", lambda s: s.ge(0), "non-negative"),
]
for table, field, predicate, rule in range_contract:
    values = pd.to_numeric(written_text[table][field], errors="coerce")
    valid = values.notna() & predicate(values)
    task2_check(
        f"VAL-RANGE-{table}-{field}", f"{table}.{field}: sensible range ({rule})",
        valid.all(), f"min={values.min()}, max={values.max()}, invalid={(~valid).sum()}",
        f"All serialized {field} values parsed numerically and checked against the semantic range.",
        "The field is numerically usable and contains no implausible value under its public meaning.",
        "Trace invalid values to parsing, percentage/currency conversion or source-quality evidence and document treatment.",
    )



**Observed result/status/interpretation:** Recorded in the executable validation register for this subsection.

### 6.2 Primary- and foreign-key checks (`VAL-PK-...`, `VAL-FK-...`)

In [47]:
fk_contract = [
    ("orders", "customer_id", "customers", "customer_id"),
    ("order_items", "order_id", "orders", "order_id"),
    ("order_items", "product_id", "products", "product_id"),
    ("deliveries", "order_id", "orders", "order_id"),
    ("product_reviews", "order_id", "orders", "order_id"),
    ("product_reviews", "order_item_id", "order_items", "order_item_id"),
    ("product_reviews", "product_id", "products", "product_id"),
    ("product_reviews", "customer_id", "customers", "customer_id"),
]
for child_table, child_field, parent_table, parent_field in fk_contract:
    orphans = set(written_text[child_table][child_field]) - set(written_text[parent_table][parent_field])
    task2_check(
        f"VAL-FK-{child_table}-{child_field}",
        f"{child_table}.{child_field} -> {parent_table}.{parent_field}",
        not orphans, f"orphans={len(orphans)}",
        f"Exact submitted {child_field} values anti-joined to {parent_table}.{parent_field}.",
        "Every child row resolves to one published parent entity.",
        "Trace orphan keys to a missing parent, incorrect identifier normalisation or a broken flattening relationship.",
    )

completed_written = set(written["orders"].loc[
    written["orders"].order_status == "Completed", "order_id"
])
delivery_order_ids = set(written["deliveries"].order_id)
task2_check(
    "VAL-GRAIN-deliveries", "one delivery per completed order",
    delivery_order_ids == completed_written and written["deliveries"].order_id.is_unique,
    f"completed_orders={len(completed_written)}, delivery_orders={len(delivery_order_ids)}",
    "Submitted orders filtered on the published Completed category and compared with unique deliveries.order_id.",
    "The delivery table has exactly the required completed-order grain.",
    "Filter canonical deliveries by completed order IDs and reconcile duplicate delivery IDs.",
)

review_item = written["product_reviews"].merge(
    written["order_items"][["order_item_id", "order_id", "product_id"]],
    on="order_item_id", suffixes=("", "_item"), validate="many_to_one"
)
review_links_ok = (
    review_item["order_id"].eq(review_item["order_id_item"]).all()
    and review_item["product_id"].eq(review_item["product_id_item"]).all()
)
task2_check(
    "VAL-REL-reviews-items", "review order/product agree with referenced order item",
    review_links_ok, f"rows_checked={len(review_item)}",
    "many-to-one merge on order_item_id, followed by order_id and product_id equality.",
    "The review-to-item one-to-many relationship preserves the item's order and product.",
    "Correct the review foreign keys from the structured source evidence; do not infer them from narrative text.",
)

review_order = written["product_reviews"].merge(
    written["orders"][["order_id", "customer_id"]],
    on="order_id", suffixes=("", "_order"), validate="many_to_one"
)
task2_check(
    "VAL-REL-reviews-orders", "review customer agrees with referenced order",
    review_order["customer_id"].eq(review_order["customer_id_order"]).all(),
    f"rows_checked={len(review_order)}",
    "many-to-one merge on order_id followed by customer_id equality.",
    "Every review belongs to the same customer as its structured order.",
    "Trace mismatches to review/order reconciliation rather than overwriting either key.",
)



**Observed result/status/interpretation:** Recorded in the executable validation register for this subsection.

### 6.3 Source coverage and reconciliation checks (`VAL-FLOW-...`)

In [48]:
expected_key_sets = {table: canonical_keys(table) for table in TASK2_PRIMARY_KEYS}
expected_key_sets["deliveries"] = set(
    canonical_deliveries_source.loc[
        canonical_deliveries_source.order_id.isin(completed_order_ids), "delivery_id"
    ]
)
for table, frame in written_text.items():
    observed_keys = set(frame[TASK2_PRIMARY_KEYS[table]])
    expected_keys = expected_key_sets[table]
    flow = overlap_profile.loc[overlap_profile.target_table == table].iloc[0]
    observed = (
        f"JSON rows/keys={flow.json_records}/{flow.json_distinct_keys}; "
        f"XML rows/keys={flow.xml_records}/{flow.xml_distinct_keys}; "
        f"shared={flow.keys_in_both_sources}; output={len(frame)}"
    )
    task2_check(
        f"VAL-FLOW-{table}", f"{table}: source/overlap row flow and canonical key coverage",
        observed_keys == expected_keys and len(frame) == len(expected_keys), observed,
        "T1-P3/T1-P6 measured within-source repeats and cross-source shared keys; T2-R1 canonical output keys.",
        "All source keys survive exactly once after data-driven reconciliation at the target grain.",
        "Inspect the duplicate/overlap profile and trace missing or extra keys through the canonical-key union.",
    )

task2_check(
    "VAL-CONFLICT-01", "no unresolved field-level conflict for a shared key",
    task2_conflicts.empty,
    f"conflicts={len(task2_conflicts)}, profile={os.path.basename(task2_conflict_path)}",
    "T2-R1 compares normalised/derived non-missing values by stable primary key and writes every disagreement to the profile.",
    "No arbitrary source precedence was needed; all shared values agree after published normalisation.",
    "Leave disagreeing fields unresolved, inspect both source values and document a justified resolution before submission.",
)



**Observed result/status/interpretation:** Recorded in the executable validation register for this subsection.

### 6.4 Arithmetic checks (`VAL-ARITH-...`)

In [49]:
line_expected = round2(written["order_items"]["quantity"] * written["order_items"]["unit_price"])
line_diff = (line_expected - written["order_items"]["line_revenue"]).abs()
task2_check(
    "VAL-MONEY-01", "line_revenue = round(quantity * unit_price, 2)",
    line_diff.le(MONEY_TOLERANCE).all(), f"max_abs_diff={line_diff.max():.6f}",
    "All submitted order-item rows; Python cent rounding and absolute tolerance 0.01.",
    "Every line follows published arithmetic step 1.",
    "Recalculate line_revenue from quantity and unit_price before aggregating orders.",
)

price_expected = round2(written["order_items"].groupby("order_id")["line_revenue"].sum())
order_price_diff = (written["orders"].set_index("order_id")["order_price"] - price_expected).abs()
task2_check(
    "VAL-MONEY-02", "order_price = sum of rounded line_revenue",
    order_price_diff.le(MONEY_TOLERANCE).all(), f"max_abs_diff={order_price_diff.max():.6f}",
    "Canonical order items grouped at order_id after line-level rounding; tolerance 0.01.",
    "Every order follows published arithmetic step 2 without duplicate-line inflation.",
    "Reconcile order-item duplicates before summing the rounded line revenues.",
)

tax_expected = round2(written["orders"]["order_price"] / 11)
tax_diff = (tax_expected - written["orders"]["tax_amount"]).abs()
task2_check(
    "VAL-MONEY-03", "tax_amount is included GST order_price / 11 before discount",
    tax_diff.le(MONEY_TOLERANCE).all(), f"max_abs_diff={tax_diff.max():.6f}",
    "All canonical orders; included-GST formula rounded to cents with tolerance 0.01.",
    "Tax is reported separately and has not been recomputed after discount.",
    "Recalculate tax_amount from undiscounted order_price and do not add it to order_total.",
)

total_expected = round2(
    written["orders"]["order_price"] * (1 - written["orders"]["coupon_discount"] / 100)
    + written["orders"]["delivery_charges"]
)
total_diff = (total_expected - written["orders"]["order_total"]).abs()
task2_check(
    "VAL-MONEY-04", "order_total follows discount then delivery sequence",
    total_diff.le(MONEY_TOLERANCE).all(), f"max_abs_diff={total_diff.max():.6f}",
    "order_price * (1 - coupon_discount/100) + delivery_charges, then cent rounding; tolerance 0.01.",
    "Published arithmetic steps 4-6 are satisfied and tax was not added again.",
    "Apply the discount to order_price, add delivery_charges, then round once to cents.",
)



**Observed result/status/interpretation:** Recorded in the executable validation register for this subsection.

### 6.5 Temporal checks (`VAL-TIME-...`)

In [50]:
# Temporal ordering among related entities.
order_delivery_time = written_text["orders"][["order_id", "order_timestamp"]].merge(
    written_text["deliveries"][["order_id", "dispatch_date", "promised_date", "delivered_date"]],
    on="order_id", validate="one_to_one"
)
order_delivery_time["order_timestamp"] = pd.to_datetime(order_delivery_time["order_timestamp"])
for column in ("dispatch_date", "promised_date", "delivered_date"):
    order_delivery_time[column] = pd.to_datetime(order_delivery_time[column])
temporal_contract = [
    ("VAL-TIME-order-dispatch", "order date <= dispatch date",
     order_delivery_time.order_timestamp.dt.normalize(), order_delivery_time.dispatch_date),
    ("VAL-TIME-dispatch-promised", "dispatch date <= promised date",
     order_delivery_time.dispatch_date, order_delivery_time.promised_date),
    ("VAL-TIME-dispatch-delivered", "dispatch date <= delivered date",
     order_delivery_time.dispatch_date, order_delivery_time.delivered_date),
]
for check_id, description, earlier, later in temporal_contract:
    violations = earlier.gt(later)
    task2_check(
        check_id, description, not violations.any(), f"rows={len(violations)}, violations={violations.sum()}",
        "Related submitted order/delivery rows joined by order_id and compared as parsed dates/timestamps.",
        "The operational event sequence is temporally valid.",
        "Inspect the source date convention and the affected relationship; document any genuine source anomaly.",
    )

review_time = written_text["product_reviews"][["review_id", "order_id", "review_timestamp"]].merge(
    order_delivery_time[["order_id", "delivered_date"]], on="order_id", validate="many_to_one"
)
review_time["review_timestamp"] = pd.to_datetime(review_time["review_timestamp"])
review_before_delivery = review_time.review_timestamp.dt.normalize().lt(review_time.delivered_date)
task2_check(
    "VAL-TIME-delivery-review", "delivered date <= review timestamp",
    not review_before_delivery.any(), f"rows={len(review_time)}, violations={review_before_delivery.sum()}",
    "Canonical reviews joined to completed deliveries by order_id.",
    "Reviews occur only after the associated product was delivered.",
    "Inspect review/order linkage and date parsing; record a genuine source anomaly rather than fabricating a date.",
)



**Observed result/status/interpretation:** Recorded in the executable validation register for this subsection.

### 6.6 Text and multilingual checks (`VAL-TEXT-...`)

In [51]:
# Reuse the submitted text function's own emoji definition so the check and the
# implementation cannot disagree. Category "So" is broader than emoji: it also
# holds (c), (R), (TM) and the degree sign, which Task 3 deliberately preserves.
from Group005_text_functions import _is_emoji_character

sku_by_product = written["products"].set_index("product_id")["product_sku"]
expected_review_sku = written["product_reviews"]["product_id"].map(sku_by_product)
task2_check(
    "VAL-TEXT-review-sku", "extracted review SKU agrees with referenced product",
    written["product_reviews"]["extracted_product_sku"].eq(expected_review_sku).all(),
    f"rows_checked={len(expected_review_sku)}",
    "Raw parser-obtained review text extracted before cleaning, then compared through product_id to products.product_sku.",
    "Every extracted SKU agrees with the structured product relationship.",
    "Inspect raw wrapper boundaries and the product foreign key; never repair the relationship from a near-match.",
)
task2_check(
    "VAL-TEXT-review-order", "extracted review order agrees with structured order_id",
    written["product_reviews"]["extracted_order_reference"].eq(
        written["product_reviews"]["order_id"]
    ).all(), f"rows_checked={len(written['product_reviews'])}",
    "Raw parser-obtained review text extracted before cleaning and compared with structured order_id.",
    "Every extracted order reference agrees with the structured review relationship.",
    "Inspect extraction boundaries and the structured source key; do not accept malformed near-matches.",
)
expected_promo = written_text["orders"]["order_id"].map(order_promo_by_id)
task2_check(
    "VAL-TEXT-promo", "promotion extraction agrees with raw structured customer_note",
    written_text["orders"]["promo_code"].eq(expected_promo).all(),
    f"rows_checked={len(expected_promo)}, literal_NaN={(expected_promo == MISSING).sum()}",
    "PROMO reference extracted from each parser-obtained raw note before narrative cleaning and reconciled by order_id.",
    "Promotion codes and literal-NaN absence follow the published extraction order.",
    "Correct the raw-value extraction boundary or resolve a recorded cross-source derivation conflict.",
)

# Cleaned narrative residue, measures and multilingual preservation.
residue_tokens = (
    "<", ">", "http://", "https://", "www.", "[system]", "[catalogue]",
    "[verified_purchase]", "[source:", "[rating:", "#verified-buyer", "@store_support",
)
for table, field in (
    ("orders", "customer_note_clean"),
    ("products", "product_description_clean"),
    ("product_reviews", "review_body_clean"),
):
    values = written_text[table][field]
    residue = values.map(lambda value: any(token in value.lower() for token in residue_tokens))
    emoji_or_symbol = values.map(lambda value: any(
        _is_emoji_character(char)
        for char in value
    ))
    lower_case = values.eq(MISSING) | values.eq(values.str.lower())
    valid = not residue.any() and not emoji_or_symbol.any() and lower_case.all()
    task2_check(
        f"VAL-TEXT-clean-{table}-{field}", f"{table}.{field}: cleaned narrative contract",
        valid,
        f"residue={residue.sum()}, emoji_or_symbol={emoji_or_symbol.sum()}, not_lower={len(values)-lower_case.sum()}",
        "Submitted narrative scanned for tags, listed markers, URLs, emoji/symbol residue and lower-case output.",
        "The designated cleaned narrative follows the published removal and case contract.",
        "Apply the published cleaning order to the parser-obtained raw value and recheck residue.",
    )

review_body = written_text["product_reviews"]["review_body_clean"]
latin_body = written_text["product_reviews"]["review_body_latin_analysis"]
expected_length = review_body.map(lambda value: 0 if value == MISSING else len(value))
expected_words = review_body.map(lambda value: 0 if value == MISSING else len(value.split()))
measure_ok = (
    written["product_reviews"]["review_length_chars"].eq(expected_length).all()
    and written["product_reviews"]["review_word_count"].eq(expected_words).all()
)
task2_check(
    "VAL-TEXT-review-measures", "review character/word measures follow sentinel behaviour",
    measure_ok, f"rows={len(review_body)}",
    "Python len and whitespace-token counts recomputed from submitted review_body_clean; literal NaN maps to zero.",
    "Review measures describe human-readable cleaned text rather than the three sentinel characters.",
    "Recompute both measures from review_body_clean with an explicit literal-NaN branch.",
)

expected_non_latin = review_body.map(contains_non_latin_script)
observed_non_latin = written_text["product_reviews"]["contains_non_latin_script"].eq("True")
latin_contains_non_latin = latin_body.map(contains_non_latin_script)
multilingual_ok = expected_non_latin.eq(observed_non_latin).all() and not latin_contains_non_latin.any()
task2_check(
    "VAL-TEXT-multilingual", "multilingual preservation and Latin-analysis separation",
    multilingual_ok,
    f"clean_non_latin={expected_non_latin.sum()}, indicator_true={observed_non_latin.sum()}, "
    f"latin_output_with_non_latin={latin_contains_non_latin.sum()}",
    "Unicode letter names/categories recomputed on both submitted review outputs; no ASCII proxy is used.",
    "Multilingual letters remain in review_body_clean, the indicator agrees, and Latin analysis removes non-Latin letters.",
    "Rebuild Latin analysis from review_body_clean and classify scripts with Unicode metadata, preserving the multilingual clean field.",
)



**Observed result/status/interpretation:** Recorded in the executable validation register for this subsection.

### 6.7 Literal `NaN` reminder

In [52]:
# Prescribed nullable strings must remain visible rather than becoming blank CSV cells.
for field in ("coupon_code", "promo_code"):
    values = written_text["orders"][field]
    malformed = values.map(lambda value: value != MISSING and extract_promo_code(value) != value)
    valid = values.ne("").all() and not malformed.any()
    task2_check(
        f"VAL-SENTINEL-{field}", f"orders.{field}: literal NaN or valid published code",
        valid, f"literal_NaN={(values == MISSING).sum()}, blank={(values == '').sum()}, malformed={malformed.sum()}",
        f"Exact-text {field} values checked with keep_default_na=False and the public promotion extractor.",
        "Every absent code is the three-character sentinel NaN; every populated code follows the public format.",
        "Replace blank/pandas missing values with literal NaN and correct malformed populated codes.",
    )



In [53]:
task2_validation = pd.DataFrame(task2_checks)
print(task2_validation.to_string(index=False))
validation_register_path = os.path.join(PROFILE_DIR, f"{GROUP_ID}_validation_register.csv")
task2_validation.to_csv(validation_register_path, index=False)
print(f"[saved] {validation_register_path}  ({len(task2_validation)} checks)")
failed_task2_checks = task2_validation[task2_validation.status != "PASS"]
passed_task2_checks = len(task2_validation) - len(failed_task2_checks)
print(f"Task 2 validation: {passed_task2_checks}/{len(task2_validation)} checks PASS")
# A validation FAIL remains visible evidence rather than being hidden by a crash.
# The register includes its proposed resolution and the unresolved value is never
# filled through source precedence.
if not failed_task2_checks.empty:
    print(f"\n{len(failed_task2_checks)} Task 2 check(s) FAIL - investigate before submission:")
    print(failed_task2_checks.to_string(index=False))

                                         check_id                                                                   description                                                                                                                                                                                                                                                                                                                                                                                                                   observed status                                                                                                                 evidence                                                                                          resolution_or_interpretation
                                      VAL-FILE-01                                             exactly six required output files                                                                                        

## 7. Export the six CSV files

The six CSV files are written in the existing `T2-O1` cell before validation so the validation register checks the serialized deliverables rather than only in-memory DataFrames.

---
#### Deliverables written by this notebook

Enumerated from disk rather than listed by hand, so the manifest cannot drift.

In [54]:
deliverables = sorted(glob.glob(os.path.join(PROFILE_DIR, f"{GROUP_ID}_T1_*.csv"))) + [MAPPING_PATH]
artefact_manifest = pd.DataFrame([
    {"artefact": path,
     "rows": len(pd.read_csv(path, keep_default_na=False)),
     "bytes": os.path.getsize(path)}
    for path in deliverables
])
print(f"{len(artefact_manifest)} artefacts written "
      f"({len(deliverables) - 1} profiling CSVs + the source-to-target mapping)")
artefact_manifest

14 artefacts written (13 profiling CSVs + the source-to-target mapping)


,artefact,rows,bytes
0,profiling/Group005_T1_assumptions_register.csv,16,5588
1,profiling/Group005_T1_categorical_domain_profile.csv,41,2545
2,profiling/Group005_T1_duplicate_and_overlap_profile.csv,6,613
3,profiling/Group005_T1_embedded_reference_profile.csv,3,455
4,profiling/Group005_T1_field_level_conflicts.csv,0,26
5,profiling/Group005_T1_format_conventions.csv,8,2201
6,profiling/Group005_T1_mapping_source_format_summary.csv,7,198
7,profiling/Group005_T1_narrative_field_profile.csv,5,599
8,profiling/Group005_T1_order_arithmetic_checks.csv,2,447
9,profiling/Group005_T1_raw_frame_profile.csv,11,818


---
#### Task 1 summary

| Question | Answer, with evidence |
|---|---|
| Are both sources parsed structurally? | Yes — `json.load` + `pandas.json_normalize`, and `BeautifulSoup(..., "lxml-xml")`, following the Week 2/3 applied sessions. No regex reconstructs document structure (`T1-P1`, `T1-P2`). |
| Is everything in pandas? | Yes — eleven raw DataFrames are built immediately after parsing, and every profiling question from `T1-P3` on is answered with pandas (`T1-P2`). |
| What are the major objects and repeated elements? | 3 JSON collections, 4 XML collections; `shoppingCart[*]` / `Shopping_Cart/Item` is the only one-to-many child of an order (`T1-P2`). |
| What is each collection's grain? | One record per order, order item, customer, delivery, product and review — but **no shared transactional key is unique within its own export**; only the single-source `customers` and `products` collections are already unique (`T1-P3`). |
| Which target fields come from where? | Customers are JSON-only, products XML-only, orders / order items / deliveries / reviews are in both (`T1-P2`, `T1-P5`, `T1-M1`). |
| How do the formats differ? | ISO vs day-first dates, native vs `Y`/`N` booleans, float vs `"AUD 1,234.56"`, integer vs `"15%"`, empty string vs empty element (`T1-P4`). |
| Does the published arithmetic hold? | All six steps reproduce the published values in both exports — but only with Python's rounding and only on de-duplicated order lines (`T1-P4`, `ASM-14`). |
| Is there duplication and overlap? | Both kinds, in every shared table; **zero field-level conflicts after normalisation**, so no source precedence rule is needed (`T1-P6`). |
| Do the relationships hold? | All eight required foreign keys resolve against the union of the two sources with zero orphans (`T1-P7`). |
| What must be assumed before transforming? | Every assumption is registered with its supporting evidence and the consequence if it is wrong (`T1-P9`, `ASM-01` onwards). |
| Is the mapping complete? | 111/111 required target fields, all six blank columns filled, asserted against the dictionary and against the profiled paths (`T1-M1`). |

**Task 3 evidence:** 18/18 supplied public cases and 61/61 student-designed cases covering
matched, unmatched, missing, multilingual and near-match inputs are demonstrated in section
3.2, together with four counts reproduced independently from the section 1 profiling.

**Not in scope here (Tasks 5 and 6):** the EDA notebook and the findings/ML sections of the
report.

## 8. Final reproducibility record

#### Task 2 summary

* Exactly six CSV files are written with dictionary field order and no helper columns.
* Shared tables are reconciled after normalisation with zero silent conflict choices.
* Order-item, order, included-GST and discounted-total arithmetic is re-derived in sequence.
* Every primary/foreign key and the order/item/review one-to-many relationships remain valid.
* Literal `NaN` is retained only for prescribed string results; numeric/boolean fields never
  receive that sentinel.